# Text detection

In [ ]:
import os
import sys
import subprocess
import importlib.util

def ensure_package(import_name: str, pip_spec: str, no_deps: bool = False):
    if importlib.util.find_spec(import_name) is None:
        cmd = [sys.executable, "-m", "pip", "install", "-q"]
        if no_deps:
            cmd.append("--no-deps")
        cmd.append(pip_spec)
        subprocess.check_call(cmd)

ensure_package("shapely", "shapely>=2.0.0")
ensure_package("pyclipper", "pyclipper>=1.3.0")
ensure_package("gdown", "gdown>=5.1.0")
ensure_package("kagglehub", "kagglehub>=0.2.5")
ensure_package("huggingface_hub", "huggingface_hub>=0.23.0")
ensure_package("einops", "einops>=0.7.0", no_deps=True)
ensure_package("vietocr", "vietocr>=0.3.11", no_deps=True)

import json
import math
import glob
import re
import shutil
import random
import zipfile
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from copy import deepcopy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as tv_models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from shapely.geometry import Polygon
from tqdm.auto import tqdm
import shapely

SEED = 20261108

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"Shapely: {shapely.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Số lượng GPU đang kích hoạt: {GPU_COUNT}")
for idx, name in enumerate(GPU_NAMES):
    print(f"GPU {idx}: {name}")

In [ ]:
# ============================================================
# CELL 2 — TẢI & THIẾT LẬP DỮ LIỆU THỰC TẾ ĐA NGUỒN
# ============================================================
WORK_DIR = Path("/kaggle/working/unified_ocr_project")
RAW_DATA_DIR = WORK_DIR / "raw_datasets"
UNIFIED_DET_DIR = WORK_DIR / "det_dataset"
UNIFIED_REC_DIR = WORK_DIR / "rec_dataset"
EXPORT_DIR = WORK_DIR / "exported_models"
KAGGLE_INPUT_DIR = Path("/kaggle/input")

VINTEXT_ROOT = RAW_DATA_DIR / "vintext_official"
ICDAR_ROOT = RAW_DATA_DIR / "icdar_vietnamese"
VIETOCR_ROOT = RAW_DATA_DIR / "vietocr_real_crops"

for folder in [WORK_DIR, RAW_DATA_DIR, UNIFIED_DET_DIR, UNIFIED_REC_DIR, EXPORT_DIR, ICDAR_ROOT, VIETOCR_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

import gdown

# 1. Tải VinText chính thức
VINTEXT_GOOGLE_DRIVE_ID = "1UUQhNvzgpZy7zXBFQp0Qox-BBjunZ0ml"

def has_vintext_gt(root: Path) -> bool:
    if not root.exists():
        return False
    try:
        return next(root.rglob("gt_*.txt"), None) is not None
    except (PermissionError, OSError):
        return False

def download_vintext():
    zip_path = RAW_DATA_DIR / "vietnamese_original.zip"
    if has_vintext_gt(VINTEXT_ROOT):
        print(f"[1/3] VinText đã có sẵn tại: {VINTEXT_ROOT}")
        return
    print("[1/3] Đang tải VinText (1.05 GB)...")
    try:
        result = gdown.download(id=VINTEXT_GOOGLE_DRIVE_ID, output=str(zip_path), quiet=False)
        if result and zip_path.exists() and zipfile.is_zipfile(zip_path):
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(VINTEXT_ROOT)
            zip_path.unlink(missing_ok=True)
            print(f" -> Đã giải nén VinText vào: {VINTEXT_ROOT}")
    except Exception as e:
        print(f" -> Lỗi tải VinText: {e}")

# 2. Tải kho Crop từ ngữ tiếng Việt thực tế của VietOCR
# Link lưu trữ bộ từ vựng crop biển báo & tài liệu thực tế
VIETOCR_DRIVE_ID = "1m3f7g4C5k1U_s14J7FkF25jVq6P3LqG0"

def download_vietocr_crops():
    crop_txt = VIETOCR_ROOT / "labels.txt"
    if crop_txt.exists() and any(VIETOCR_ROOT.rglob("*.jpg")):
        print(f"[2/3] Dữ liệu VietOCR Crop thực tế đã sẵn sàng tại: {VIETOCR_ROOT}")
        return
    
    zip_path = RAW_DATA_DIR / "vietocr_data.zip"
    print("[2/3] Đang tải kho crop chữ tiếng Việt thực tế từ VietOCR Benchmark...")
    try:
        # Thử tải bộ crop chữ thực tế
        result = gdown.download(id=VIETOCR_DRIVE_ID, output=str(zip_path), quiet=True)
        if result and zip_path.exists() and zipfile.is_zipfile(zip_path):
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(VIETOCR_ROOT)
            zip_path.unlink(missing_ok=True)
            print(f" -> Đã nạp thành công kho crop chữ VietOCR vào {VIETOCR_ROOT}")
        else:
            if zip_path.exists():
                zip_path.unlink()
            print(" -> Link drive phụ bị bận, tiếp tục với các nguồn thay thế...")
    except Exception as e:
        print(f" -> Bỏ qua VietOCR drive ({e}).")

# 3. Kéo tập ICDAR / Scene Text tiếng Việt từ kho raw GitHub của các nhóm nghiên cứu
def download_vietnamese_scene_text_icdar():
    target_dir = ICDAR_ROOT / "vietnamese_scene"
    if target_dir.exists() and any(target_dir.rglob("*.txt")):
        print(f"[3/3] Dữ liệu Scene Text bổ sung đã sẵn sàng tại: {ICDAR_ROOT}")
        return
    
    print("[3/3] Đang kéo tập nhãn & tọa độ Scene Text tiếng Việt mở rộng...")
    # Tải bộ chú thích ICDAR MLT Vietnamese mở rộng
    repo_url = "https://raw.githubusercontent.com/Fsoft-AIC/open-vietnamese-scene-text/main/annotations.zip"
    zip_path = RAW_DATA_DIR / "scene_text_extra.zip"
    try:
        subprocess.run(["wget", "-q", repo_url, "-O", str(zip_path)], check=False, timeout=60)
        if zip_path.exists() and zip_path.stat().st_size > 5000:
            target_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(target_dir)
            zip_path.unlink(missing_ok=True)
            print(f" -> Đã giải nén dữ liệu mở rộng vào {target_dir}")
        else:
            if zip_path.exists():
                zip_path.unlink()
    except Exception:
        pass

download_vintext()
download_vietocr_crops()
download_vietnamese_scene_text_icdar()
print("Hoàn tất thiết lập tải dữ liệu.")

In [ ]:
# ============================================================
# CELL 3 — BỘ TỪ VỰNG TIẾNG VIỆT 233 LỚP KÝ TỰ & GIẢI MÃ NFC
# ============================================================
IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)

VIETOCR_CHARS = (
    "aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬbBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆ"
    "fFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoOòÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢpPqQrRsStT"
    "uUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰvVwWxXyYỳỲỷỶỹỸýÝỵỴzZ0123456789"
    "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "
)

VOCAB_RAW = list(" !\"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~")
PAD_IDS = {96, 108}

ACCENT_MAP = {
    95: {"d": "đ", "D": "Đ"},
    100: {"a": "ă", "A": "Ă"},
    101: {"a": "â", "A": "Â", "e": "ê", "E": "Ê", "o": "ô", "O": "Ô"},
    102: {"o": "ơ", "O": "Ơ", "u": "ư", "U": "Ư"},
}

TONE_MAP = {
    103: {
        "a": "à", "ă": "ằ", "â": "ầ", "e": "è", "ê": "ề", "i": "ì", "o": "ò", "ô": "ồ",
        "ơ": "ờ", "u": "ù", "ư": "ừ", "y": "ỳ", "A": "À", "Ă": "Ằ", "Â": "Ầ", "E": "È",
        "Ê": "Ề", "I": "Ì", "O": "Ò", "Ô": "Ồ", "Ơ": "Ờ", "U": "Ù", "Ư": "Ừ", "Y": "Ỳ",
    },
    104: {
        "a": "ả", "ă": "ẳ", "â": "ẩ", "e": "ẻ", "ê": "ể", "i": "ỉ", "o": "ỏ", "ô": "ổ",
        "ơ": "ở", "u": "ủ", "ư": "ử", "y": "ỷ", "A": "Ả", "Ă": "Ẳ", "Â": "Ẩ", "E": "Ẻ",
        "Ê": "Ể", "I": "Ỉ", "O": "Ỏ", "Ô": "Ổ", "Ơ": "Ở", "U": "Ủ", "Ư": "Ử", "Y": "Ỷ",
    },
    105: {
        "a": "ã", "ă": "ẵ", "â": "ẫ", "e": "ẽ", "ê": "ễ", "i": "ĩ", "o": "õ", "ô": "ỗ",
        "ơ": "ỡ", "u": "ũ", "ư": "ữ", "y": "ỹ", "A": "Ã", "Ă": "Ẵ", "Â": "Ẫ", "E": "Ẽ",
        "Ê": "Ễ", "I": "Ĩ", "O": "Õ", "Ô": "Ỗ", "Ơ": "Ỡ", "U": "Ũ", "Ư": "Ữ", "Y": "Ỹ",
    },
    106: {
        "a": "á", "ă": "ắ", "â": "ấ", "e": "é", "ê": "ế", "i": "í", "o": "ó", "ô": "ố",
        "ơ": "ớ", "u": "ú", "ư": "ứ", "y": "ý", "A": "Á", "Ă": "Ắ", "Â": "Ấ", "E": "É",
        "Ê": "Ế", "I": "Í", "O": "Ó", "Ô": "Ố", "Ơ": "Ớ", "U": "Ú", "Ư": "Ứ", "Y": "Ý",
    },
    107: {
        "a": "ạ", "ă": "ặ", "â": "ậ", "e": "ẹ", "ê": "ệ", "i": "ị", "o": "ọ", "ô": "ộ",
        "ơ": "ợ", "u": "ụ", "ư": "ự", "y": "ỵ", "A": "Ạ", "Ă": "Ặ", "Â": "Ậ", "E": "Ẹ",
        "Ê": "Ệ", "I": "Ị", "O": "Ọ", "Ô": "Ộ", "Ơ": "Ợ", "U": "Ụ", "Ư": "Ự", "Y": "Ỵ",
    },
}

def normalize_nfc(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("\ufeff", "").replace("\u200b", "").replace("\t", " ").replace("\n", " ").replace("\r", " ")
    return " ".join(text.split()).strip()

def clean_and_decode_rec(raw_rec) -> str:
    sequence = [int(v) for v in raw_rec]
    while sequence and sequence[-1] in PAD_IDS:
        sequence.pop()
    chars = []
    for token_id in sequence:
        if token_id in PAD_IDS or token_id < 0:
            continue
        if 0 <= token_id < len(VOCAB_RAW):
            chars.append(VOCAB_RAW[token_id])
        elif token_id in ACCENT_MAP and chars:
            chars[-1] = ACCENT_MAP[token_id].get(chars[-1], chars[-1])
        elif token_id in TONE_MAP and chars:
            chars[-1] = TONE_MAP[token_id].get(chars[-1], chars[-1])
    return normalize_nfc("".join(chars))

class VietnameseVocab:
    def __init__(self, characters=None):
        self.PAD, self.SOS, self.EOS, self.UNK = "<pad>", "<sos>", "<eos>", "<unk>"
        self.special_tokens = [self.PAD, self.SOS, self.EOS, self.UNK]
        self.char2idx = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.idx2char = {idx: token for token, idx in self.char2idx.items()}
        for ch in (VIETOCR_CHARS if characters is None else characters):
            if ch not in self.char2idx:
                idx = len(self.char2idx)
                self.char2idx[ch] = idx
                self.idx2char[idx] = ch

    def encode(self, text: str) -> list[int]:
        norm = normalize_nfc(text)
        return [
            self.char2idx[self.SOS],
            *(self.char2idx.get(ch, self.char2idx[self.UNK]) for ch in norm),
            self.char2idx[self.EOS],
        ]

    def decode(self, indices) -> str:
        chars = []
        for val in indices:
            idx = int(val)
            if idx in (self.char2idx[self.PAD], self.char2idx[self.EOS]):
                break
            if idx in (self.char2idx[self.SOS], self.char2idx[self.UNK]):
                continue
            chars.append(self.idx2char.get(idx, ""))
        return normalize_nfc("".join(chars))

    def to_json(self) -> dict:
        return {
            "special_tokens": self.special_tokens,
            "characters": [self.idx2char[idx] for idx in range(4, len(self))],
        }

    def __len__(self) -> int:
        return len(self.char2idx)

VOCAB = VietnameseVocab()
(EXPORT_DIR / "vocab.json").write_text(json.dumps(VOCAB.to_json(), ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Đã khởi tạo bộ từ vựng với {len(VOCAB)} lớp ký tự.")

In [ ]:
import hashlib
import json
import re
from pathlib import Path

import cv2
import numpy as np


USE_EXTERNAL_DETECTION_DATA = False
VALIDATION_RATIO = 0.10


def normalize_detection_polygon(coords, min_area: float = 0.5):
    points = np.asarray(coords, dtype=np.float32)
    if points.ndim == 1:
        if points.size == 4:
            x1, y1, x2, y2 = points
            points = np.asarray(
                [[x1, y1], [x2, y1], [x2, y2], [x1, y2]],
                dtype=np.float32,
            )
        elif points.size >= 6 and points.size % 2 == 0:
            points = points.reshape(-1, 2)
        else:
            return None

    if points.ndim != 2 or points.shape[1] != 2 or len(points) < 3:
        return None
    if not np.isfinite(points).all():
        return None

    points = points.reshape(-1, 2).copy()
    if len(points) > 3 and np.allclose(points[0], points[-1]):
        points = points[:-1]
    if len(points) < 3:
        return None
    if cv2.contourArea(points.reshape(-1, 1, 2)) < min_area:
        return None
    return points.astype(np.float32)


def order_points_clockwise(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32).reshape(4, 2)
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    ordered = points[np.argsort(angles)]
    start = int(np.argmin(ordered.sum(axis=1)))
    return np.roll(ordered, -start, axis=0)


def crop_and_rectify(
    image: np.ndarray,
    polygon_points: np.ndarray,
    pad_ratio: float = 0.06,
) -> np.ndarray | None:
    points = np.asarray(polygon_points, dtype=np.float32).reshape(-1, 2)
    if image is None or image.ndim != 3 or len(points) < 3:
        return None

    rectangle = cv2.minAreaRect(points)
    (center_x, center_y), (rect_width, rect_height), angle = rectangle
    if rect_width < 2 or rect_height < 2:
        return None

    padding = max(0.0, float(pad_ratio))
    rect_width *= 1.0 + 2.0 * padding
    rect_height *= 1.0 + 2.0 * padding
    box = order_points_clockwise(
        cv2.boxPoints(((center_x, center_y), (rect_width, rect_height), angle))
    )

    crop_width = int(
        max(np.linalg.norm(box[0] - box[1]), np.linalg.norm(box[2] - box[3]))
    )
    crop_height = int(
        max(np.linalg.norm(box[0] - box[3]), np.linalg.norm(box[1] - box[2]))
    )
    if crop_width < 6 or crop_height < 6:
        return None

    destination = np.asarray(
        [
            [0, 0],
            [crop_width - 1, 0],
            [crop_width - 1, crop_height - 1],
            [0, crop_height - 1],
        ],
        dtype=np.float32,
    )
    transform = cv2.getPerspectiveTransform(box, destination)
    crop = cv2.warpPerspective(
        image,
        transform,
        (crop_width, crop_height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )
    if crop.shape[0] > crop.shape[1] * 1.5:
        crop = cv2.rotate(crop, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return crop


def _vintext_numeric_id(name: str) -> str:
    stem = Path(str(name)).stem.lower()
    match = re.search(r"(\d+)$", stem)
    if match:
        return str(int(match.group(1)))
    for prefix in ("gt_", "gt-", "image_", "img_", "im_", "img", "im"):
        if stem.startswith(prefix):
            return stem[len(prefix):].lstrip("_-")
    return stem


def _parse_vintext_line(line: str):
    fields = line.strip().lstrip("\ufeff").split(",")
    coordinates = []
    split_at = None
    for index, field in enumerate(fields):
        try:
            coordinates.append(float(field.strip()))
        except ValueError:
            split_at = index
            break

    if split_at is None or len(coordinates) < 6 or len(coordinates) % 2:
        return None, ""

    polygon = normalize_detection_polygon(coordinates)
    text = normalize_nfc(",".join(fields[split_at:]))
    return polygon, text or "###"


def _image_md5(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def _split_from_directory_name(name: str):
    normalized = name.lower()
    if normalized in {"train_images", "train_image", "training", "train"}:
        return "train"
    if normalized in {
        "val_image", "val_images", "validation_images", "valid_images", "validation", "val"
    }:
        return "val"
    if normalized in {"test_image", "test_images", "test"}:
        return "test"
    if normalized in {"unseen_test_images", "unseen_test", "unseen-test"}:
        return "unseen_test"
    return None


def discover_vintext_layouts(search_roots: list[Path]) -> list[dict]:
    layouts = []
    seen_roots = set()
    image_directory_names = {
        "train_images", "train_image", "training", "train",
        "val_image", "val_images", "validation_images", "valid_images", "validation", "val",
        "test_image", "test_images", "test",
        "unseen_test_images", "unseen_test",
    }
    label_directory_names = {"labels", "label", "gt", "annotations"}

    for search_root in search_roots:
        if not search_root.exists():
            continue
        candidates = [search_root] + [path for path in search_root.rglob("*") if path.is_dir()]
        for parent in candidates:
            try:
                child_directories = [path for path in parent.iterdir() if path.is_dir()]
            except OSError:
                continue
            label_dir = next(
                (path for path in child_directories if path.name.lower() in label_directory_names),
                None,
            )
            split_dirs = {
                split: path
                for path in child_directories
                if path.name.lower() in image_directory_names
                for split in [_split_from_directory_name(path.name)]
                if split is not None
            }
            if label_dir is None or "train" not in split_dirs:
                continue

            root_key = str(parent.resolve())
            if root_key in seen_roots:
                continue
            seen_roots.add(root_key)
            layouts.append(
                {
                    "root": parent,
                    "labels": label_dir,
                    "split_dirs": split_dirs,
                }
            )
    return layouts


def _parse_vintext_layout(layout: dict):
    label_map = {}
    for label_path in layout["labels"].rglob("gt_*.txt"):
        label_map[_vintext_numeric_id(label_path.stem)] = label_path

    detection_items = []
    recognition_items = []
    allowed_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

    for split_name, image_dir in layout["split_dirs"].items():
        for image_path in sorted(image_dir.rglob("*")):
            if not image_path.is_file() or image_path.suffix.lower() not in allowed_extensions:
                continue
            label_path = label_map.get(_vintext_numeric_id(image_path.stem))
            if label_path is None:
                continue

            annotations = []
            try:
                lines = label_path.read_text(
                    encoding="utf-8-sig", errors="replace"
                ).splitlines()
            except OSError:
                continue

            for line in lines:
                polygon, text = _parse_vintext_line(line)
                if polygon is None:
                    continue
                annotations.append(
                    {
                        "transcription": text,
                        "points": polygon.tolist(),
                    }
                )
                if text != "###" and text and all(
                    character in VOCAB.char2idx for character in text
                ):
                    recognition_items.append(
                        {
                            "source_image": str(image_path.resolve()),
                            "polygon": polygon.tolist(),
                            "text": text,
                            "split": split_name,
                            "domain": "real",
                        }
                    )

            if annotations:
                resolved_path = image_path.resolve()
                detection_items.append(
                    {
                        "image_path": str(resolved_path),
                        "annotations": annotations,
                        "split": split_name,
                        "domain": "real",
                        "image_md5": _image_md5(resolved_path),
                    }
                )

    return detection_items, recognition_items


def _parse_external_text_files(raw_data_dir: Path):
    detection_items = []
    recognition_items = []
    for annotation_path in raw_data_dir.rglob("*.txt"):
        if annotation_path.name.lower().startswith("gt_"):
            continue
        try:
            lines = annotation_path.read_text(
                encoding="utf-8-sig", errors="replace"
            ).splitlines()
        except OSError:
            continue
        if not lines or "\t" not in lines[0]:
            continue

        path_parts = {part.lower() for part in annotation_path.parts}
        file_name = annotation_path.stem.lower()
        source_string = str(annotation_path).lower()
        is_synthetic = "synth" in source_string
        if any(part in {"unseen_test", "unseen_test_images"} for part in path_parts) or "unseen_test" in file_name:
            split_name = "unseen_test"
        elif any(part in {"val", "valid", "validation", "val_images", "validation_images"} for part in path_parts) or file_name.startswith(("val", "valid", "validation")):
            split_name = "val"
        elif any(part in {"test", "testing", "test_image", "test_images"} for part in path_parts) or file_name.startswith("test"):
            split_name = "test"
        else:
            split_name = "train"

        parent = annotation_path.parent
        for line in lines:
            if "\t" not in line:
                continue
            relative_path, label = line.split("\t", 1)
            image_path = (parent / relative_path).resolve()
            if not image_path.is_file():
                image_path = (parent.parent / relative_path).resolve()
            if not image_path.is_file():
                continue

            label = label.strip()
            if label.startswith("["):
                try:
                    raw_annotations = json.loads(label)
                except json.JSONDecodeError:
                    continue

                annotations = []
                for annotation in raw_annotations:
                    polygon = normalize_detection_polygon(annotation.get("points", []))
                    if polygon is None:
                        continue
                    text = normalize_nfc(
                        annotation.get("transcription", "###")
                    ) or "###"
                    annotations.append(
                        {"transcription": text, "points": polygon.tolist()}
                    )
                if annotations:
                    detection_items.append(
                        {
                            "image_path": str(image_path),
                            "annotations": annotations,
                            "split": split_name,
                            "domain": "synth" if is_synthetic else "real",
                            "image_md5": _image_md5(image_path),
                        }
                    )
            else:
                text = normalize_nfc(label)
                if text and text != "###" and all(
                    character in VOCAB.char2idx for character in text
                ):
                    recognition_items.append(
                        {
                            "crop_path": str(image_path),
                            "text": text,
                            "split": split_name,
                            "domain": "synth" if is_synthetic else "real",
                        }
                    )
    return detection_items, recognition_items


def _deduplicate_detection_items(items: list[dict]):
    split_priority = {"train": 0, "val": 1, "unseen_test": 2, "test": 3}
    grouped = {}
    for item in items:
        image_path = Path(item["image_path"])
        if not image_path.is_file():
            continue
        image_hash = item.get("image_md5") or _image_md5(image_path)
        item["image_md5"] = image_hash
        grouped.setdefault(image_hash, []).append(item)

    kept = []
    duplicate_count = 0
    cross_split_conflicts = []
    for image_hash, group in grouped.items():
        group.sort(
            key=lambda item: (
                -split_priority.get(item.get("split", "train"), 0),
                item["image_path"],
            )
        )
        winner = group[0]
        kept.append(winner)
        if len(group) > 1:
            duplicate_count += len(group) - 1
            splits = sorted({item.get("split", "train") for item in group})
            if len(splits) > 1:
                cross_split_conflicts.append(
                    {"md5": image_hash, "kept_split": winner["split"], "removed_splits": splits}
                )
    return kept, duplicate_count, cross_split_conflicts


def _make_fixed_train_validation_split(
    train_items: list[dict], validation_ratio: float = 0.1
):
    if len(train_items) < 2:
        raise ValueError("Cần ít nhất 2 ảnh Train để tạo Validation")
    ranked = sorted(
        train_items,
        key=lambda item: hashlib.sha256(item["image_md5"].encode("ascii")).hexdigest(),
    )
    validation_count = max(1, int(round(len(ranked) * validation_ratio)))
    validation_items = ranked[:validation_count]
    remaining_train = ranked[validation_count:]
    return remaining_train, validation_items


def parse_all_datasets(
    validation_ratio: float = VALIDATION_RATIO,
    use_external_detection_data: bool = USE_EXTERNAL_DETECTION_DATA,
):
    search_roots = [Path(KAGGLE_INPUT_DIR), Path(RAW_DATA_DIR)]
    detection_items = []
    recognition_items = []
    discovered_layouts = discover_vintext_layouts(search_roots)
    explicit_validation_found = False

    for layout in discovered_layouts:
        parsed_detection, parsed_recognition = _parse_vintext_layout(layout)
        detection_items.extend(parsed_detection)
        recognition_items.extend(parsed_recognition)
        explicit_validation_found |= "val" in layout["split_dirs"]

    if use_external_detection_data:
        external_detection, external_recognition = _parse_external_text_files(
            Path(RAW_DATA_DIR)
        )
        detection_items.extend(external_detection)
        recognition_items.extend(external_recognition)

    detection_items, duplicate_count, cross_split_conflicts = (
        _deduplicate_detection_items(detection_items)
    )

    train_items = [item for item in detection_items if item["split"] == "train"]
    validation_items = [item for item in detection_items if item["split"] == "val"]
    test_items = [item for item in detection_items if item["split"] == "test"]
    unseen_test_items = [
        item for item in detection_items if item["split"] == "unseen_test"
    ]

    if not validation_items:
        train_items, validation_items = _make_fixed_train_validation_split(
            train_items, validation_ratio=validation_ratio
        )
        validation_source = "Tách cố định từ Train"
    else:
        validation_source = "Thư mục Validation được cung cấp"

    md5_to_split = {}
    for split_name, split_items in (
        ("train", train_items),
        ("val", validation_items),
        ("test", test_items),
        ("unseen_test", unseen_test_items),
    ):
        for item in split_items:
            item["split"] = split_name
            md5_to_split[item["image_md5"]] = split_name
    filtered_recognition = []
    seen_recognition = set()
    for item in recognition_items:
        source_image = item.get("source_image")
        if source_image and Path(source_image).is_file():
            source_hash = _image_md5(Path(source_image))
            kept_split = md5_to_split.get(source_hash)
            if kept_split is not None:
                item["split"] = kept_split
        polygon_key = tuple(
            np.asarray(item.get("polygon", []), dtype=np.float32).reshape(-1).round(2)
        )
        key = (
            item.get("source_image") or item.get("crop_path"),
            polygon_key,
            item["text"],
            item["split"],
        )
        if key not in seen_recognition:
            seen_recognition.add(key)
            filtered_recognition.append(item)

    split_hashes = {
        "train": {item["image_md5"] for item in train_items},
        "validation": {item["image_md5"] for item in validation_items},
        "test": {item["image_md5"] for item in test_items},
        "unseen_test": {item["image_md5"] for item in unseen_test_items},
    }
    split_names = list(split_hashes)
    for index, first_name in enumerate(split_names):
        for second_name in split_names[index + 1:]:
            overlap = split_hashes[first_name] & split_hashes[second_name]
            if overlap:
                raise RuntimeError(
                    f"Rò rỉ ảnh giữa {first_name} và {second_name}: {len(overlap)} ảnh"
                )

    global DET_UNSEEN_TEST_ITEMS, DET_DATA_AUDIT
    DET_UNSEEN_TEST_ITEMS = unseen_test_items
    DET_DATA_AUDIT = {
        "vintext_layouts": [str(layout["root"]) for layout in discovered_layouts],
        "validation_source": validation_source,
        "validation_ratio": validation_ratio if validation_source.startswith("Tách") else None,
        "duplicate_images_removed": duplicate_count,
        "cross_split_conflicts": cross_split_conflicts,
        "explicit_validation_found": explicit_validation_found,
        "use_external_detection_data": use_external_detection_data,
        "counts": {
            "train": len(train_items),
            "validation": len(validation_items),
            "test": len(test_items),
            "unseen_test": len(unseen_test_items),
        },
    }

    print("\n========== KIỂM TOÁN DỮ LIỆU DETECTION ==========")
    print(f"Nguồn Validation: {validation_source}")
    print(f"Dữ liệu Detection ngoài VinText: {'Có' if use_external_detection_data else 'Không'}")
    print(f"Ảnh Train: {len(train_items):,}")
    print(f"Ảnh Validation: {len(validation_items):,}")
    print(f"Ảnh Test giữ riêng: {len(test_items):,}")
    print(f"Ảnh Unseen Test giữ riêng: {len(unseen_test_items):,}")
    print(f"Ảnh trùng đã loại: {duplicate_count:,}")
    print(f"Xung đột MD5 giữa các split: {len(cross_split_conflicts):,}")
    print(f"Mẫu Recognition: {len(filtered_recognition):,}")
    print("=================================================\n")

    return (
        train_items,
        validation_items,
        filtered_recognition,
        test_items,
        unseen_test_items,
    )


(
    DET_TRAIN_ITEMS,
    DET_VAL_ITEMS,
    RAW_REC_ITEMS,
    DET_TEST_ITEMS,
    DET_UNSEEN_TEST_ITEMS,
) = parse_all_datasets()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from torchvision.models.resnet import Bottleneck
from torchvision.ops import DeformConv2d


class ModulatedDeformConv2d(nn.Module):
    def __init__(self, source_conv: nn.Conv2d):
        super().__init__()
        kernel_height, kernel_width = source_conv.kernel_size
        kernel_points = kernel_height * kernel_width

        self.offset_mask = nn.Conv2d(
            source_conv.in_channels,
            3 * kernel_points,
            kernel_size=source_conv.kernel_size,
            stride=source_conv.stride,
            padding=source_conv.padding,
            dilation=source_conv.dilation,
            bias=True,
        )
        self.deform = DeformConv2d(
            source_conv.in_channels,
            source_conv.out_channels,
            kernel_size=source_conv.kernel_size,
            stride=source_conv.stride,
            padding=source_conv.padding,
            dilation=source_conv.dilation,
            groups=source_conv.groups,
            bias=source_conv.bias is not None,
        )
        self.kernel_points = kernel_points

        with torch.no_grad():
            self.deform.weight.copy_(source_conv.weight)
            if source_conv.bias is not None:
                self.deform.bias.copy_(source_conv.bias)
            nn.init.zeros_(self.offset_mask.weight)
            nn.init.zeros_(self.offset_mask.bias)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        offset_mask = self.offset_mask(value)
        offset_channels = 2 * self.kernel_points
        offsets = offset_mask[:, :offset_channels]
        mask_logits = offset_mask[:, offset_channels:]
        modulation = 2.0 * torch.sigmoid(mask_logits)
        return self.deform(value, offsets, mask=modulation)


class ResNetBackbone(nn.Module):
    def __init__(
        self,
        model_name: str = "resnet50",
        pretrained: bool = True,
        freeze_early: bool = False,
        use_dcn: bool = False,
    ):
        super().__init__()
        backbones = {
            "resnet18": (tv_models.resnet18, tv_models.ResNet18_Weights, [64, 128, 256, 512]),
            "resnet34": (tv_models.resnet34, tv_models.ResNet34_Weights, [64, 128, 256, 512]),
            "resnet50": (tv_models.resnet50, tv_models.ResNet50_Weights, [256, 512, 1024, 2048]),
        }
        if model_name not in backbones:
            raise ValueError(f"Backbone không được hỗ trợ: {model_name}")

        constructor, weights_enum, self.out_channels = backbones[model_name]
        resnet = constructor(weights=weights_enum.DEFAULT if pretrained else None)

        if use_dcn:
            if model_name != "resnet50":
                raise ValueError("DCNv2 hiện chỉ hỗ trợ ResNet50")
            for stage in (resnet.layer2, resnet.layer3, resnet.layer4):
                for block in stage:
                    if isinstance(block, Bottleneck):
                        block.conv2 = ModulatedDeformConv2d(block.conv2)

        self.stem = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
        )
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.use_dcn = bool(use_dcn)
        self.freeze_early = bool(freeze_early and pretrained)

        if self.freeze_early:
            for module in (self.stem, self.layer1):
                for parameter in module.parameters():
                    parameter.requires_grad_(False)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_early:
            self.stem.eval()
            self.layer1.eval()
        return self

    def forward(self, image: torch.Tensor) -> list[torch.Tensor]:
        value = self.stem(image)
        c2 = self.layer1(value)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return [c2, c3, c4, c5]


class AdaptiveScaleFusion(nn.Module):
    def __init__(
        self,
        in_channels: list[int],
        inner_channels: int = 256,
        dropout: float = 0.1,
        use_asf_spatial: bool = True,
    ):
        super().__init__()
        if inner_channels % 4 != 0:
            raise ValueError("inner_channels phải chia hết cho 4")
        stage_channels = inner_channels // 4

        self.in2 = nn.Conv2d(in_channels[0], inner_channels, 1, bias=False)
        self.in3 = nn.Conv2d(in_channels[1], inner_channels, 1, bias=False)
        self.in4 = nn.Conv2d(in_channels[2], inner_channels, 1, bias=False)
        self.in5 = nn.Conv2d(in_channels[3], inner_channels, 1, bias=False)

        def output_block():
            return nn.Sequential(
                nn.Conv2d(inner_channels, stage_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(stage_channels),
                nn.ReLU(inplace=True),
            )

        self.out2 = output_block()
        self.out3 = output_block()
        self.out4 = output_block()
        self.out5 = output_block()
        self.attn_conv = nn.Sequential(
            nn.Conv2d(inner_channels, inner_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout),
        )
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(inner_channels, stage_channels, 3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(stage_channels, 1, 1),
            nn.Sigmoid(),
        )
        self.scale_conv = nn.Sequential(
            nn.Conv2d(inner_channels, stage_channels, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(stage_channels, 4, 1),
        )
        self.use_asf_spatial = bool(use_asf_spatial)

    def forward(self, features: list[torch.Tensor]) -> torch.Tensor:
        c2, c3, c4, c5 = features
        p5 = self.in5(c5)
        p4 = self.in4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="nearest")
        p3 = self.in3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="nearest")
        p2 = self.in2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="nearest")

        target_size = c2.shape[-2:]
        f2 = self.out2(p2)
        f3 = F.interpolate(self.out3(p3), size=target_size, mode="nearest")
        f4 = F.interpolate(self.out4(p4), size=target_size, mode="nearest")
        f5 = F.interpolate(self.out5(p5), size=target_size, mode="nearest")

        attention = self.attn_conv(torch.cat([f2, f3, f4, f5], dim=1))
        spatial_weight = self.spatial_conv(attention)
        guided = attention * (1.0 + spatial_weight)
        scale_logits = self.scale_conv(guided)
        if not self.use_asf_spatial:
            scale_logits = F.adaptive_avg_pool2d(scale_logits, 1)
        scale_weights = torch.softmax(scale_logits, dim=1)

        fused = torch.cat(
            [
                f2 * scale_weights[:, 0:1],
                f3 * scale_weights[:, 1:2],
                f4 * scale_weights[:, 2:3],
                f5 * scale_weights[:, 3:4],
            ],
            dim=1,
        )
        return fused


class DBHead(nn.Module):
    def __init__(
        self,
        in_channels: int = 256,
        k: int = 50,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.k = float(k)
        inner_channels = in_channels // 4

        self.binarize = nn.Sequential(
            nn.Conv2d(in_channels, inner_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout),
            nn.ConvTranspose2d(inner_channels, inner_channels, 2, stride=2, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(inner_channels, 1, 2, stride=2),
        )
        self.thresh = nn.Sequential(
            nn.Conv2d(in_channels, inner_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout),
            nn.ConvTranspose2d(inner_channels, inner_channels, 2, stride=2, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(inner_channels, 1, 2, stride=2),
        )
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
        nn.init.constant_(self.binarize[-1].bias, -2.0)
        nn.init.constant_(self.thresh[-1].bias, -1.0)

    def forward(self, features: torch.Tensor) -> dict[str, torch.Tensor]:
        prob_map = torch.sigmoid(self.binarize(features).float())
        thresh_map = torch.sigmoid(self.thresh(features).float())
        binary_map = torch.sigmoid(self.k * (prob_map - thresh_map))
        return {
            "prob_map": prob_map,
            "thresh_map": thresh_map,
            "binary_map": binary_map,
        }


class DBNetPlusPlus(nn.Module):
    def __init__(
        self,
        model_name: str = "resnet50",
        pretrained: bool = True,
        freeze_early: bool = False,
        inner_channels: int = 256,
        k: int = 50,
        dropout: float = 0.15,
        use_dcn: bool = False,
        use_asf_spatial: bool = True,
    ):
        super().__init__()
        self.config = {
            "model_name": model_name,
            "freeze_early": freeze_early,
            "inner_channels": inner_channels,
            "k": k,
            "dropout": dropout,
            "use_dcn": use_dcn,
            "use_asf_spatial": use_asf_spatial,
        }
        self.backbone = ResNetBackbone(
            model_name=model_name,
            pretrained=pretrained,
            freeze_early=freeze_early,
            use_dcn=use_dcn,
        )
        self.neck = AdaptiveScaleFusion(
            in_channels=self.backbone.out_channels,
            inner_channels=inner_channels,
            dropout=dropout * 0.5,
            use_asf_spatial=use_asf_spatial,
        )
        self.head = DBHead(
            in_channels=inner_channels,
            k=k,
            dropout=dropout,
        )

    def get_param_groups(
        self,
        base_lr: float,
        backbone_lr_mult: float = 0.2,
        weight_decay: float = 3e-4,
    ) -> list[dict]:
        backbone_decay, backbone_no_decay = [], []
        neck_head_decay, neck_head_no_decay = [], []

        for name, parameter in self.named_parameters():
            if not parameter.requires_grad:
                continue
            no_decay = parameter.ndim <= 1 or name.endswith(".bias")
            if name.startswith("backbone."):
                (backbone_no_decay if no_decay else backbone_decay).append(parameter)
            else:
                (neck_head_no_decay if no_decay else neck_head_decay).append(parameter)

        groups = []
        if backbone_decay:
            groups.append(
                {
                    "params": backbone_decay,
                    "lr": base_lr * backbone_lr_mult,
                    "weight_decay": weight_decay,
                }
            )
        if backbone_no_decay:
            groups.append(
                {
                    "params": backbone_no_decay,
                    "lr": base_lr * backbone_lr_mult,
                    "weight_decay": 0.0,
                }
            )
        if neck_head_decay:
            groups.append(
                {"params": neck_head_decay, "lr": base_lr, "weight_decay": weight_decay}
            )
        if neck_head_no_decay:
            groups.append(
                {"params": neck_head_no_decay, "lr": base_lr, "weight_decay": 0.0}
            )
        return groups

    def forward(self, image: torch.Tensor) -> dict[str, torch.Tensor]:
        return self.head(self.neck(self.backbone(image)))


class DBLoss(nn.Module):
    def __init__(
        self,
        alpha: float = 5.0,
        beta: float = 1.0,
        gamma: float = 10.0,
        negative_ratio: int = 3,
        label_smoothing: float = 0.0,
        min_negative_pixels: int = 100,
        eps: float = 1e-6,
    ):
        super().__init__()
        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.negative_ratio = int(negative_ratio)
        self.label_smoothing = float(label_smoothing)
        self.min_negative_pixels = int(min_negative_pixels)
        self.eps = float(eps)
        self.last_metrics: dict[str, float] = {}

    @staticmethod
    def _ensure_4d(tensor: torch.Tensor) -> torch.Tensor:
        return tensor.unsqueeze(1) if tensor.ndim == 3 else tensor

    def _balanced_bce_loss(
        self,
        prediction: torch.Tensor,
        target: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        prediction = prediction.float().clamp(self.eps, 1.0 - self.eps)
        target = target.float()
        valid_mask = mask.float() > 0.5
        positive_mask = (target > 0.5) & valid_mask
        negative_mask = (target <= 0.5) & valid_mask

        smoothed_target = (
            target * (1.0 - self.label_smoothing)
            + 0.5 * self.label_smoothing
        )
        element_loss = F.binary_cross_entropy(
            prediction, smoothed_target, reduction="none"
        )
        positive_loss = element_loss[positive_mask]
        negative_loss = element_loss[negative_mask]

        positive_count = int(positive_mask.sum().item())
        negative_total = int(negative_mask.sum().item())
        if positive_count == 0:
            negative_count = min(negative_total, self.min_negative_pixels)
        else:
            negative_count = min(
                negative_total,
                positive_count * self.negative_ratio,
            )

        if negative_count > 0:
            negative_loss, _ = torch.topk(
                negative_loss.reshape(-1), k=negative_count
            )
        else:
            negative_loss = negative_loss[:0]

        denominator = float(positive_count + negative_count) + self.eps
        return (positive_loss.sum() + negative_loss.sum()) / denominator

    def _dice_loss(
        self,
        prediction: torch.Tensor,
        target: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        prediction = prediction.float()
        target = target.float()
        valid_mask = mask.float()
        intersection = (prediction * target * valid_mask).sum()
        union = (prediction * valid_mask).sum() + (target * valid_mask).sum()
        return 1.0 - (2.0 * intersection + self.eps) / (union + self.eps)

    def _l1_loss(
        self,
        prediction: torch.Tensor,
        target: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        prediction = prediction.float()
        target = target.float()
        valid_mask = mask.float()
        denominator = valid_mask.sum()
        if denominator.item() <= 0:
            return prediction.sum() * 0.0
        return (torch.abs(prediction - target) * valid_mask).sum() / (
            denominator + self.eps
        )

    def forward(
        self,
        predictions: dict[str, torch.Tensor],
        targets: dict[str, torch.Tensor] | torch.Tensor,
        gt_mask: torch.Tensor | None = None,
        gt_thresh: torch.Tensor | None = None,
        thresh_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        if isinstance(targets, dict):
            gt_prob = targets.get(
                "prob_map", targets.get("gt", targets.get("shrink_map"))
            )
            gt_mask = targets.get(
                "prob_mask", targets.get("mask", targets.get("shrink_mask"))
            )
            gt_thresh = targets.get("thresh_map")
            thresh_mask = targets.get("thresh_mask")
        else:
            gt_prob = targets

        if gt_prob is None:
            raise ValueError("Không tìm thấy shrink map trong target")

        gt_prob = self._ensure_4d(gt_prob)
        gt_mask = self._ensure_4d(gt_mask) if gt_mask is not None else torch.ones_like(gt_prob)
        gt_thresh = self._ensure_4d(gt_thresh) if gt_thresh is not None else torch.zeros_like(gt_prob)
        thresh_mask = self._ensure_4d(thresh_mask) if thresh_mask is not None else torch.zeros_like(gt_prob)

        loss_prob = self._balanced_bce_loss(
            predictions["prob_map"], gt_prob, gt_mask
        )
        loss_binary = self._dice_loss(
            predictions["binary_map"], gt_prob, gt_mask
        )
        loss_thresh = self._l1_loss(
            predictions["thresh_map"], gt_thresh, thresh_mask
        )
        total_loss = (
            self.alpha * loss_prob
            + self.beta * loss_binary
            + self.gamma * loss_thresh
        )
        self.last_metrics = {
            "loss_prob": float(loss_prob.detach().item()),
            "loss_binary": float(loss_binary.detach().item()),
            "loss_thresh": float(loss_thresh.detach().item()),
        }
        return total_loss


def verify_modulated_dcn_initialization() -> float:
    source_conv = nn.Conv2d(8, 12, kernel_size=3, padding=1, bias=True)
    deform_conv = ModulatedDeformConv2d(source_conv)
    source_conv.eval()
    deform_conv.eval()
    input_tensor = torch.randn(2, 8, 24, 24)
    with torch.inference_mode():
        reference = source_conv(input_tensor)
        actual = deform_conv(input_tensor)
    max_error = float((reference - actual).abs().max().item())
    torch.testing.assert_close(actual, reference, rtol=1e-5, atol=1e-5)
    return max_error


DBNet = DBNetPlusPlus
DBNetPP = DBNetPlusPlus

DCN_INITIALIZATION_MAX_ERROR = verify_modulated_dcn_initialization()
print(
    f"Kiểm tra khởi tạo DCNv2 hoàn tất; sai số lớn nhất: "
    f"{DCN_INITIALIZATION_MAX_ERROR:.3e}"
)


In [ ]:
import math
import random

import cv2
import numpy as np
import pyclipper
import torch
from shapely.geometry import Polygon, box as shapely_box
from torch.utils.data import Dataset

cv2.setNumThreads(0)


def _to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


def _extract_sample_fields(item: dict):
    image_path = next(
        (
            str(item[key])
            for key in ("image_path", "img_path", "path", "image", "file_path", "filename")
            if item.get(key)
        ),
        None,
    )
    if image_path is None:
        raise KeyError(f"Không tìm thấy đường dẫn ảnh: {list(item.keys())}")

    raw_polygons = next(
        (
            item[key]
            for key in ("polygons", "polys", "boxes", "bboxes", "points")
            if item.get(key) is not None
        ),
        None,
    )
    raw_texts = next(
        (
            item[key]
            for key in ("texts", "transcriptions", "labels", "words_text")
            if item.get(key) is not None
        ),
        None,
    )
    raw_ignores = next(
        (
            item[key]
            for key in ("ignores", "ignore", "dontcare")
            if item.get(key) is not None
        ),
        None,
    )

    if raw_polygons is None and "annotations" in item:
        annotations = item["annotations"]
        raw_polygons = [
            annotation.get("points", annotation.get("polygon", annotation.get("bbox")))
            for annotation in annotations
        ]
        raw_texts = [
            annotation.get("text", annotation.get("transcription", ""))
            for annotation in annotations
        ]
        raw_ignores = [
            annotation.get("ignore", annotation.get("dontcare", False))
            for annotation in annotations
        ]

    polygons, ignores = [], []
    for index, raw_polygon in enumerate(raw_polygons if raw_polygons is not None else []):
        polygon = normalize_detection_polygon(raw_polygon, min_area=0.5)
        if polygon is None:
            continue

        is_ignore = False
        if raw_ignores is not None and index < len(raw_ignores):
            is_ignore = _to_bool(raw_ignores[index])
        if raw_texts is not None and index < len(raw_texts):
            text = normalize_nfc(raw_texts[index])
            is_ignore = is_ignore or text in {"", "###", "*"}

        polygons.append(polygon.astype(np.float32, copy=True))
        ignores.append(bool(is_ignore))

    return image_path, polygons, ignores


def _offset_polygon_fast(polygon: np.ndarray, offset: float) -> list[np.ndarray]:
    points = np.asarray(polygon, dtype=np.float64).reshape(-1, 2)
    if len(points) < 3 or not np.isfinite(points).all():
        return []

    scale = 100.0
    integer_path = np.round(points * scale).astype(np.int64).tolist()
    try:
        simple_paths = pyclipper.SimplifyPolygon(
            integer_path, pyclipper.PFT_NONZERO
        )
        results = []
        for path in simple_paths:
            if len(path) < 3:
                continue
            offsetter = pyclipper.PyclipperOffset()
            offsetter.AddPath(
                path, pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON
            )
            for result in offsetter.Execute(float(offset) * scale):
                if len(result) >= 3:
                    results.append(np.asarray(result, dtype=np.float32) / scale)
        return results
    except pyclipper.Error:
        return []


def _point_to_polygon_distance(
    points: np.ndarray,
    grid_x: np.ndarray,
    grid_y: np.ndarray,
) -> np.ndarray:
    x1 = points[:, 0][:, None, None]
    y1 = points[:, 1][:, None, None]
    x2 = np.roll(points[:, 0], -1)[:, None, None]
    y2 = np.roll(points[:, 1], -1)[:, None, None]
    dx = x2 - x1
    dy = y2 - y1
    length_squared = np.maximum(dx * dx + dy * dy, 1e-6)
    ratio = np.clip(
        ((grid_x[None] - x1) * dx + (grid_y[None] - y1) * dy)
        / length_squared,
        0.0,
        1.0,
    )
    projected_x = x1 + ratio * dx
    projected_y = y1 + ratio * dy
    distance_squared = (
        (grid_x[None] - projected_x) ** 2
        + (grid_y[None] - projected_y) ** 2
    )
    return np.sqrt(distance_squared.min(axis=0))


def _draw_minimum_core(
    shrink_map: np.ndarray,
    polygon: np.ndarray,
    valid_height: int,
    valid_width: int,
) -> bool:
    points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
    x1 = max(0, int(np.floor(points[:, 0].min())) - 2)
    y1 = max(0, int(np.floor(points[:, 1].min())) - 2)
    x2 = min(valid_width - 1, int(np.ceil(points[:, 0].max())) + 2)
    y2 = min(valid_height - 1, int(np.ceil(points[:, 1].max())) + 2)
    if x2 < x1 or y2 < y1:
        return False

    local_mask = np.zeros((y2 - y1 + 1, x2 - x1 + 1), dtype=np.uint8)
    local_points = np.round(points - np.asarray([x1, y1])).astype(np.int32)
    cv2.fillPoly(local_mask, [local_points], 1)
    if not np.any(local_mask):
        center = np.mean(points, axis=0)
        cx = int(np.clip(round(center[0]), 0, valid_width - 1))
        cy = int(np.clip(round(center[1]), 0, valid_height - 1))
        shrink_map[cy, cx] = 1.0
        return True

    distance = cv2.distanceTransform(local_mask, cv2.DIST_L2, 3)
    maximum = float(distance.max())
    if maximum > 0:
        core = (distance >= max(0.5, maximum * 0.5)) & (local_mask > 0)
    else:
        core = local_mask > 0

    if not np.any(core):
        y_local, x_local = np.unravel_index(int(distance.argmax()), distance.shape)
        core[y_local, x_local] = True

    region = shrink_map[y1:y2 + 1, x1:x2 + 1]
    region[core] = 1.0
    shrink_map[y1:y2 + 1, x1:x2 + 1] = region
    return True


class UnifiedDetectionDataset(Dataset):
    def __init__(
        self,
        items: list[dict],
        target_size: int = 960,
        is_train: bool = True,
        shrink_ratio: float = 0.4,
        thresh_min: float = 0.3,
        thresh_max: float = 0.7,
        min_text_size: float = 3.0,
    ):
        self.items = list(items)
        self.target_size = int(target_size)
        self.is_train = bool(is_train)
        self.shrink_ratio = float(shrink_ratio)
        self.thresh_min = float(thresh_min)
        self.thresh_max = float(thresh_max)
        self.min_text_size = float(min_text_size)
        self.mean = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)
        self.target_audit = {
            "valid_polygons": 0,
            "ignore_polygons": 0,
            "standard_shrink": 0,
            "fallback_core": 0,
            "invalid_after_resize": 0,
        }

    def __len__(self) -> int:
        return len(self.items)

    def _apply_geometric_augmentations(
        self,
        image: np.ndarray,
        polygons: list[np.ndarray],
        ignores: list[bool],
    ):
        height, width = image.shape[:2]

        if random.random() < 0.7:
            angle = random.uniform(-12.0, 12.0)
            scale = random.uniform(0.85, 1.15)
            center = (width * 0.5, height * 0.5)
            matrix = cv2.getRotationMatrix2D(center, angle, scale)
            cosine = abs(matrix[0, 0])
            sine = abs(matrix[0, 1])
            new_width = max(1, int(height * sine + width * cosine))
            new_height = max(1, int(height * cosine + width * sine))
            matrix[0, 2] += new_width * 0.5 - center[0]
            matrix[1, 2] += new_height * 0.5 - center[1]
            image = cv2.warpAffine(
                image,
                matrix,
                (new_width, new_height),
                flags=cv2.INTER_LINEAR,
                borderValue=(114, 114, 114),
            )
            polygons = [
                (
                    np.hstack(
                        [polygon, np.ones((len(polygon), 1), dtype=np.float32)]
                    )
                    @ matrix.T
                ).astype(np.float32)
                for polygon in polygons
            ]
            height, width = image.shape[:2]

        if random.random() < 0.35 and width > 64 and height > 64:
            delta_x = width * 0.04
            delta_y = height * 0.04
            source = np.asarray(
                [[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]],
                dtype=np.float32,
            )
            destination = np.asarray(
                [
                    [random.uniform(0, delta_x), random.uniform(0, delta_y)],
                    [width - 1 - random.uniform(0, delta_x), random.uniform(0, delta_y)],
                    [width - 1 - random.uniform(0, delta_x), height - 1 - random.uniform(0, delta_y)],
                    [random.uniform(0, delta_x), height - 1 - random.uniform(0, delta_y)],
                ],
                dtype=np.float32,
            )
            transform = cv2.getPerspectiveTransform(source, destination)
            image = cv2.warpPerspective(
                image,
                transform,
                (width, height),
                flags=cv2.INTER_LINEAR,
                borderValue=(114, 114, 114),
            )
            polygons = [
                cv2.perspectiveTransform(
                    polygon.reshape(-1, 1, 2), transform
                ).reshape(-1, 2).astype(np.float32)
                for polygon in polygons
            ]

        if random.random() < 0.65 and width > 128 and height > 128:
            crop_scale = random.uniform(0.35, 0.90)
            crop_width = min(width, max(64, int(width * crop_scale)))
            crop_height = min(height, max(64, int(height * crop_scale)))
            selected_crop = None

            for _ in range(20):
                x1 = random.randint(0, width - crop_width)
                y1 = random.randint(0, height - crop_height)
                x2, y2 = x1 + crop_width, y1 + crop_height
                cuts_text = False

                for polygon, is_ignore in zip(polygons, ignores):
                    if is_ignore:
                        continue
                    px1, py1 = polygon.min(axis=0)
                    px2, py2 = polygon.max(axis=0)
                    intersects = (
                        min(x2, px2) > max(x1, px1)
                        and min(y2, py2) > max(y1, py1)
                    )
                    contained = (
                        px1 >= x1 and py1 >= y1 and px2 <= x2 and py2 <= y2
                    )
                    if intersects and not contained:
                        cuts_text = True
                        break

                if not cuts_text:
                    selected_crop = (x1, y1, x2, y2)
                    break

            if selected_crop is not None:
                x1, y1, x2, y2 = selected_crop
                image = image[y1:y2, x1:x2]
                crop_rectangle = shapely_box(0, 0, x2 - x1, y2 - y1)
                cropped_polygons, cropped_ignores = [], []

                for polygon, is_ignore in zip(polygons, ignores):
                    shifted = polygon - np.asarray([x1, y1], dtype=np.float32)
                    shape = Polygon(shifted).buffer(0)
                    if shape.is_empty or shape.area <= 0.5:
                        continue
                    clipped = shape.intersection(crop_rectangle)
                    if clipped.is_empty or clipped.area <= 0.5:
                        continue
                    if clipped.geom_type == "MultiPolygon":
                        clipped = max(clipped.geoms, key=lambda part: part.area)
                    if clipped.geom_type != "Polygon":
                        continue

                    clipped_points = np.asarray(
                        clipped.exterior.coords[:-1], dtype=np.float32
                    )
                    if len(clipped_points) < 3:
                        continue
                    truncated = clipped.area / max(shape.area, 1e-6) < 0.98
                    cropped_polygons.append(clipped_points)
                    cropped_ignores.append(bool(is_ignore or truncated))

                polygons, ignores = cropped_polygons, cropped_ignores
                height, width = image.shape[:2]

        if random.random() < 0.5:
            image = cv2.flip(image, 1)
            flipped_polygons = []
            for polygon in polygons:
                flipped = polygon.copy()
                flipped[:, 0] = width - 1.0 - flipped[:, 0]
                flipped_polygons.append(flipped)
            polygons = flipped_polygons

        return image, polygons, ignores

    def _apply_photometric_augmentations(self, image: np.ndarray) -> np.ndarray:
        if random.random() < 0.25:
            kernel_size = random.choice([3, 5, 7])
            angle = random.uniform(0.0, 180.0)
            kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
            kernel[kernel_size // 2, :] = 1.0
            rotation = cv2.getRotationMatrix2D(
                (kernel_size * 0.5 - 0.5, kernel_size * 0.5 - 0.5),
                angle,
                1.0,
            )
            kernel = cv2.warpAffine(kernel, rotation, (kernel_size, kernel_size))
            if kernel.sum() > 0:
                image = cv2.filter2D(image, -1, kernel / kernel.sum())

        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 0] = (hsv[:, :, 0] + random.uniform(-8.0, 8.0)) % 180.0
            hsv[:, :, 1] = np.clip(
                hsv[:, :, 1] * random.uniform(0.75, 1.25), 0, 255
            )
            hsv[:, :, 2] = np.clip(
                hsv[:, :, 2] * random.uniform(0.8, 1.2), 0, 255
            )
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        return image

    def _resize_and_pad(self, image: np.ndarray, polygons: list[np.ndarray]):
        height, width = image.shape[:2]
        scale = self.target_size / float(max(height, width))
        resized_width = max(1, int(round(width * scale)))
        resized_height = max(1, int(round(height * scale)))
        resized = cv2.resize(
            image,
            (resized_width, resized_height),
            interpolation=cv2.INTER_LINEAR,
        )
        canvas = np.full(
            (self.target_size, self.target_size, 3), 114, dtype=np.uint8
        )
        canvas[:resized_height, :resized_width] = resized
        scaled_polygons = [polygon * scale for polygon in polygons]
        return canvas, scaled_polygons, resized_height, resized_width

    def _build_targets(
        self,
        polygons: list[np.ndarray],
        ignores: list[bool],
        valid_height: int,
        valid_width: int,
    ):
        size = self.target_size
        shrink_map = np.zeros((size, size), dtype=np.float32)
        training_mask = np.zeros((size, size), dtype=np.float32)
        training_mask[:valid_height, :valid_width] = 1.0
        threshold_map = np.full((size, size), self.thresh_min, dtype=np.float32)
        threshold_mask = np.zeros((size, size), dtype=np.float32)

        for source_polygon, is_ignore in zip(polygons, ignores):
            polygon = np.asarray(source_polygon, dtype=np.float32).reshape(-1, 2).copy()
            if len(polygon) < 3 or not np.isfinite(polygon).all():
                continue

            polygon[:, 0] = np.clip(polygon[:, 0], 0, valid_width - 1)
            polygon[:, 1] = np.clip(polygon[:, 1], 0, valid_height - 1)
            polygon_int = np.round(polygon).astype(np.int32)
            if cv2.contourArea(polygon) <= 0.5:
                self.target_audit["invalid_after_resize"] += 1
                continue

            if is_ignore:
                self.target_audit["ignore_polygons"] += 1
                cv2.fillPoly(training_mask, [polygon_int], 0.0)
                continue

            self.target_audit["valid_polygons"] += 1
            area = float(cv2.contourArea(polygon))
            perimeter = float(cv2.arcLength(polygon, closed=True))
            if area <= 0.5 or perimeter <= 0.0:
                self.target_audit["invalid_after_resize"] += 1
                _draw_minimum_core(
                    shrink_map, polygon, valid_height, valid_width
                )
                continue

            minimum_side = min(cv2.minAreaRect(polygon)[1])
            distance = area * (1.0 - self.shrink_ratio ** 2) / max(perimeter, 1e-6)
            shrunk_polygons = []
            if minimum_side >= self.min_text_size and distance > 0.5:
                shrunk_polygons = _offset_polygon_fast(polygon, -distance)
                if not shrunk_polygons:
                    shrunk_polygons = _offset_polygon_fast(
                        polygon, -distance * 0.5
                    )

            shrink_pixels = 0
            for shrunk in shrunk_polygons:
                if len(shrunk) < 3:
                    continue
                shrunk_int = np.round(shrunk).astype(np.int32)
                if cv2.contourArea(shrunk_int) <= 0:
                    continue
                cv2.fillPoly(shrink_map, [shrunk_int], 1.0)
                shrink_pixels += 1

            if shrink_pixels > 0:
                self.target_audit["standard_shrink"] += 1
            else:
                _draw_minimum_core(
                    shrink_map, polygon, valid_height, valid_width
                )
                self.target_audit["fallback_core"] += 1

            if distance <= 0.5:
                continue

            expanded_polygons = _offset_polygon_fast(polygon, distance)
            if expanded_polygons:
                expanded = max(
                    expanded_polygons,
                    key=lambda points: cv2.contourArea(points),
                )
            else:
                expanded = polygon

            x1 = max(0, int(np.floor(expanded[:, 0].min())))
            y1 = max(0, int(np.floor(expanded[:, 1].min())))
            x2 = min(valid_width - 1, int(np.ceil(expanded[:, 0].max())))
            y2 = min(valid_height - 1, int(np.ceil(expanded[:, 1].max())))
            if x2 < x1 or y2 < y1:
                continue

            local_mask = np.zeros(
                (y2 - y1 + 1, x2 - x1 + 1), dtype=np.uint8
            )
            local_points = np.round(
                expanded - np.asarray([x1, y1], dtype=np.float32)
            ).astype(np.int32)
            cv2.fillPoly(local_mask, [local_points], 1)
            if not np.any(local_mask):
                continue

            grid_x, grid_y = np.meshgrid(
                np.arange(x1, x2 + 1, dtype=np.float32),
                np.arange(y1, y2 + 1, dtype=np.float32),
            )
            distances = _point_to_polygon_distance(polygon, grid_x, grid_y)
            normalized_distance = np.clip(
                1.0 - distances / max(distance, 1e-4), 0.0, 1.0
            )
            target_values = self.thresh_min + normalized_distance * (
                self.thresh_max - self.thresh_min
            )
            active = local_mask > 0

            threshold_patch = threshold_map[y1:y2 + 1, x1:x2 + 1]
            threshold_patch[active] = np.maximum(
                threshold_patch[active], target_values[active]
            )
            threshold_map[y1:y2 + 1, x1:x2 + 1] = threshold_patch

            mask_patch = threshold_mask[y1:y2 + 1, x1:x2 + 1]
            mask_patch[active] = 1.0
            threshold_mask[y1:y2 + 1, x1:x2 + 1] = mask_patch

        return shrink_map, training_mask, threshold_map, threshold_mask

    def __getitem__(self, index: int):
        item = self.items[index]
        image_path, polygons, ignores = _extract_sample_fields(item)
        bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError(f"Không đọc được ảnh: {image_path}")
        image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        if self.is_train:
            image, polygons, ignores = self._apply_geometric_augmentations(
                image, polygons, ignores
            )
            image = self._apply_photometric_augmentations(image)

        canvas, scaled_polygons, valid_height, valid_width = self._resize_and_pad(
            image, polygons
        )
        image_tensor = (canvas.astype(np.float32) / 255.0 - self.mean) / self.std
        image_tensor = torch.from_numpy(image_tensor.transpose(2, 0, 1)).float()

        if not self.is_train:
            valid_polygons = []
            ignore_polygons = []
            for polygon, is_ignore in zip(scaled_polygons, ignores):
                points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
                if len(points) < 3 or not np.isfinite(points).all():
                    continue
                if is_ignore:
                    ignore_polygons.append(points)
                else:
                    valid_polygons.append(points)
            return (
                image_tensor,
                valid_polygons,
                ignore_polygons,
                image_path,
                valid_height,
                valid_width,
            )

        targets = self._build_targets(
            scaled_polygons,
            ignores,
            valid_height,
            valid_width,
        )
        return (
            image_tensor,
            torch.from_numpy(targets[0]),
            torch.from_numpy(targets[1]),
            torch.from_numpy(targets[2]),
            torch.from_numpy(targets[3]),
        )


def det_val_collate(batch):
    images = torch.stack([sample[0] for sample in batch], dim=0)
    valid_polygons = [sample[1] for sample in batch]
    ignore_polygons = [sample[2] for sample in batch]
    image_paths = [sample[3] for sample in batch]
    valid_sizes = [(sample[4], sample[5]) for sample in batch]
    return images, valid_polygons, ignore_polygons, image_paths, valid_sizes


In [ ]:
import gc
import itertools
import json
import math
import random
import sys
from copy import deepcopy
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pyclipper
import torch
import torch.nn as nn
import torch.nn.functional as F
from shapely.geometry import Polygon
from torch.utils.data import DataLoader

cv2.setNumThreads(0)


DETECTION_CONFIG = {
    "target_size": 1024,
    "min_text_size": 3.0,
    "use_dcn": True,
    "use_asf_spatial": True,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "batch_norm_momentum": 0.05,
    "epochs": 100,
    "patience": 25,
    "min_delta": 1e-4,
    "base_lr": 3e-4,
    "backbone_lr_multiplier": 0.2,
    "weight_decay": 3e-4,
    "warmup_epochs": 3,
    "num_workers": 2,
    "train_scales": (896, 960, 1024),
    "run_threshold_sweep": True,
    "resume_path": None,
    "postprocess": {
        "thresh": 0.25,
        "box_thresh": 0.50,
        "unclip_ratio": 1.4,
    },
}


BEST_DET_CKPT = EXPORT_DIR / "dbnetpp_v3_best_weights.pth"
BEST_DET_BUNDLE = EXPORT_DIR / "dbnetpp_v3_best_checkpoint.pth"
LAST_DET_CKPT = EXPORT_DIR / "dbnetpp_v3_last_checkpoint.pth"
BEST_POSTPROCESS_CONFIG = dict(DETECTION_CONFIG["postprocess"])


def _clear_cuda_memory():
    for attr in ("last_traceback", "last_value", "last_type"):
        if hasattr(sys, attr):
            setattr(sys, attr, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.995):
        self.ema = deepcopy(model).eval()
        self.decay = float(decay)
        self.updates = 0
        for parameter in self.ema.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        self.updates += 1
        decay = min(self.decay, (1.0 + self.updates) / (10.0 + self.updates))
        model_state = model.state_dict()
        for name, ema_value in self.ema.state_dict().items():
            model_value = model_state[name].detach()
            if ema_value.dtype.is_floating_point:
                ema_value.mul_(decay).add_(model_value, alpha=1.0 - decay)
            else:
                ema_value.copy_(model_value)


def polygon_iou(first_polygon, second_polygon) -> float:
    try:
        first = (
            first_polygon
            if isinstance(first_polygon, Polygon)
            else Polygon(np.asarray(first_polygon, dtype=np.float64)).buffer(0)
        )
        second = (
            second_polygon
            if isinstance(second_polygon, Polygon)
            else Polygon(np.asarray(second_polygon, dtype=np.float64)).buffer(0)
        )
        if first.is_empty or second.is_empty or first.area <= 0 or second.area <= 0:
            return 0.0
        union_area = first.union(second).area
        if union_area <= 0:
            return 0.0
        return float(first.intersection(second).area / union_area)
    except Exception:
        return 0.0


def _valid_shapely_polygon(points):
    try:
        shape = Polygon(np.asarray(points, dtype=np.float64).reshape(-1, 2)).buffer(0)
        if shape.is_empty or shape.area <= 0:
            return None
        if shape.geom_type == "MultiPolygon":
            shape = max(shape.geoms, key=lambda part: part.area)
        if shape.geom_type != "Polygon" or shape.area <= 0:
            return None
        return shape
    except Exception:
        return None


def _offset_detection_polygon(points: np.ndarray, distance: float) -> list[np.ndarray]:
    polygon = np.asarray(points, dtype=np.float64).reshape(-1, 2)
    if len(polygon) < 3 or not np.isfinite(polygon).all():
        return []

    scale = 100.0
    path = np.round(polygon * scale).astype(np.int64).tolist()
    try:
        if not pyclipper.Orientation(path):
            path.reverse()
        offsetter = pyclipper.PyclipperOffset()
        offsetter.AddPath(path, pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
        expanded = offsetter.Execute(float(distance) * scale)
        return [
            np.asarray(item, dtype=np.float32) / scale
            for item in expanded
            if len(item) >= 3
        ]
    except pyclipper.Error:
        return []


def seg_to_polygons(
    prob_map: np.ndarray,
    thresh: float = 0.3,
    box_thresh: float = 0.55,
    unclip_ratio: float = 1.5,
    min_size: float = 2.0,
    max_candidates: int = 2000,
    approx_epsilon_ratio: float = 0.002,
    return_stats: bool = False,
):
    probability = np.asarray(prob_map, dtype=np.float32)
    if probability.ndim != 2:
        raise ValueError("prob_map phải có kích thước H x W")

    map_height, map_width = probability.shape
    bitmap = (probability > float(thresh)).astype(np.uint8)
    contours, _ = cv2.findContours(
        bitmap, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    polygons, scores = [], []
    score_candidates = 0

    for contour in contours[:max_candidates]:
        if len(contour) < 3:
            continue
        perimeter = float(cv2.arcLength(contour, closed=True))
        if perimeter <= 0:
            continue

        epsilon = max(0.25, approx_epsilon_ratio * perimeter)
        approximated = cv2.approxPolyDP(
            contour, epsilon, closed=True
        ).reshape(-1, 2)
        if len(approximated) < 3:
            approximated = contour.reshape(-1, 2)
        if len(approximated) < 3:
            continue

        shape = _valid_shapely_polygon(approximated)
        if shape is None or shape.area < max(2.0, min_size * min_size):
            continue

        x_min, y_min, x_max, y_max = shape.bounds
        x1 = max(0, int(np.floor(x_min)))
        y1 = max(0, int(np.floor(y_min)))
        x2 = min(map_width - 1, int(np.ceil(x_max)))
        y2 = min(map_height - 1, int(np.ceil(y_max)))
        if x2 < x1 or y2 < y1:
            continue

        local_mask = np.zeros(
            (y2 - y1 + 1, x2 - x1 + 1), dtype=np.uint8
        )
        local_points = np.round(
            np.asarray(shape.exterior.coords[:-1], dtype=np.float32)
            - np.asarray([x1, y1], dtype=np.float32)
        ).astype(np.int32)
        cv2.fillPoly(local_mask, [local_points], 1)
        if not np.any(local_mask):
            continue

        score = float(
            cv2.mean(
                probability[y1:y2 + 1, x1:x2 + 1],
                local_mask,
            )[0]
        )
        score_candidates += 1
        if score < box_thresh:
            continue

        offset_distance = shape.area * float(unclip_ratio) / max(shape.length, 1e-6)
        expanded_polygons = _offset_detection_polygon(
            np.asarray(shape.exterior.coords[:-1], dtype=np.float32),
            offset_distance,
        )
        if not expanded_polygons:
            continue

        for expanded in expanded_polygons:
            expanded_shape = _valid_shapely_polygon(expanded)
            if expanded_shape is None:
                continue
            output_points = np.asarray(
                expanded_shape.exterior.coords[:-1], dtype=np.float32
            )
            output_points[:, 0] = np.clip(output_points[:, 0], 0, map_width - 1)
            output_points[:, 1] = np.clip(output_points[:, 1], 0, map_height - 1)
            if len(output_points) < 3:
                continue
            polygons.append(output_points)
            scores.append(score)

    stats = {
        "contours_before_filter": int(len(contours)),
        "score_candidates": int(score_candidates),
        "polygons_after_filter": int(len(polygons)),
    }
    if return_stats:
        return polygons, scores, stats
    return polygons, scores


def _prepare_multiscale_train_batch(
    images: torch.Tensor,
    shrink_map: torch.Tensor,
    training_mask: torch.Tensor,
    threshold_map: torch.Tensor,
    threshold_mask: torch.Tensor,
    scales: tuple[int, ...] = (1024,),
):
    target_scale = int(random.choice(scales))
    current_size = images.shape[-2:]
    if current_size != (target_scale, target_scale):
        images = F.interpolate(
            images,
            size=(target_scale, target_scale),
            mode="bilinear",
            align_corners=False,
        )
        shrink_map = F.interpolate(
            shrink_map.unsqueeze(1),
            size=(target_scale, target_scale),
            mode="nearest",
        ).squeeze(1)
        training_mask = F.interpolate(
            training_mask.unsqueeze(1),
            size=(target_scale, target_scale),
            mode="nearest",
        ).squeeze(1)
        threshold_map = F.interpolate(
            threshold_map.unsqueeze(1),
            size=(target_scale, target_scale),
            mode="nearest",
        ).squeeze(1)
        threshold_mask = F.interpolate(
            threshold_mask.unsqueeze(1),
            size=(target_scale, target_scale),
            mode="nearest",
        ).squeeze(1)

    if random.random() < 0.25:
        images = images + torch.randn_like(images) * random.uniform(0.01, 0.03)

    if random.random() < 0.35:
        batch_size, _, height, width = images.shape
        cut_height = min(height, random.randint(24, 64))
        cut_width = min(width, random.randint(24, 64))
        y1 = random.randint(0, max(0, height - cut_height))
        x1 = random.randint(0, max(0, width - cut_width))
        has_text = (
            shrink_map[:, y1:y1 + cut_height, x1:x1 + cut_width]
            .amax(dim=(1, 2)) > 0.1
        )
        safe_batch = (~has_text).view(batch_size, 1, 1, 1)
        patch = images[:, :, y1:y1 + cut_height, x1:x1 + cut_width]
        images[:, :, y1:y1 + cut_height, x1:x1 + cut_width] = torch.where(
            safe_batch, torch.zeros_like(patch), patch
        )

    return images, shrink_map, training_mask, threshold_map, threshold_mask


def _short_side(points) -> float:
    array = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    if len(array) < 3:
        return 0.0
    return float(min(cv2.minAreaRect(array)[1]))


def _size_group(points) -> str:
    side = _short_side(points)
    if side <= 10.0:
        return "small"
    if side <= 24.0:
        return "medium"
    return "large"


def _match_image_predictions(
    predicted_polygons: list[np.ndarray],
    prediction_scores: list[float],
    ground_truth_polygons: list[np.ndarray],
    ignore_polygons: list[np.ndarray],
    iou_threshold: float = 0.5,
    ignore_iou_threshold: float = 0.5,
    ignore_prediction_coverage: float = 0.5,
):
    ground_truth_shapes = [
        _valid_shapely_polygon(polygon) for polygon in ground_truth_polygons
    ]
    ignore_shapes = [
        shape
        for shape in (_valid_shapely_polygon(polygon) for polygon in ignore_polygons)
        if shape is not None
    ]
    matched_ground_truth = set()
    prediction_status = ["FP"] * len(predicted_polygons)
    matches = []
    ignored_predictions = 0

    prediction_order = sorted(
        range(len(predicted_polygons)),
        key=lambda index: float(prediction_scores[index]),
        reverse=True,
    )

    for prediction_index in prediction_order:
        prediction_shape = _valid_shapely_polygon(
            predicted_polygons[prediction_index]
        )
        if prediction_shape is None:
            prediction_status[prediction_index] = "FP"
            continue

        best_iou = 0.0
        best_ground_truth = -1
        for ground_truth_index, ground_truth_shape in enumerate(ground_truth_shapes):
            if ground_truth_index in matched_ground_truth or ground_truth_shape is None:
                continue
            overlap = polygon_iou(prediction_shape, ground_truth_shape)
            if overlap > best_iou:
                best_iou = overlap
                best_ground_truth = ground_truth_index

        if best_ground_truth >= 0 and best_iou >= iou_threshold:
            matched_ground_truth.add(best_ground_truth)
            prediction_status[prediction_index] = "TP"
            matches.append(
                {
                    "prediction_index": prediction_index,
                    "ground_truth_index": best_ground_truth,
                    "iou": best_iou,
                }
            )
            continue

        prediction_area = float(prediction_shape.area)
        ignored = False
        for ignore_shape in ignore_shapes:
            intersection = prediction_shape.intersection(ignore_shape).area
            if prediction_area <= 0:
                continue
            prediction_coverage = intersection / prediction_area
            union_area = prediction_shape.union(ignore_shape).area
            ignore_iou = intersection / max(union_area, 1e-9)
            if (
                ignore_iou >= ignore_iou_threshold
                or prediction_coverage > ignore_prediction_coverage
            ):
                ignored = True
                break

        if ignored:
            prediction_status[prediction_index] = "IGNORE"
            ignored_predictions += 1
        else:
            prediction_status[prediction_index] = "FP"

    false_negative_indices = [
        index
        for index, shape in enumerate(ground_truth_shapes)
        if shape is not None and index not in matched_ground_truth
    ]
    return {
        "statuses": prediction_status,
        "matches": matches,
        "matched_ground_truth": matched_ground_truth,
        "false_negative_indices": false_negative_indices,
        "ignored_predictions": ignored_predictions,
    }


def _metric_triplet(true_positives: int, false_positives: int, false_negatives: int):
    precision_denominator = true_positives + false_positives
    recall_denominator = true_positives + false_negatives
    precision = true_positives / precision_denominator if precision_denominator else 0.0
    recall = true_positives / recall_denominator if recall_denominator else 0.0
    hmean = (
        2.0 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )
    return precision, recall, hmean


def _evaluate_one_record(
    record: dict,
    postprocess: dict,
    iou_threshold: float = 0.5,
):
    probability_map = np.asarray(record["prob_map"], dtype=np.float32)
    predicted_polygons, prediction_scores, decode_stats = seg_to_polygons(
        probability_map,
        thresh=postprocess["thresh"],
        box_thresh=postprocess["box_thresh"],
        unclip_ratio=postprocess["unclip_ratio"],
        return_stats=True,
    )
    match_result = _match_image_predictions(
        predicted_polygons,
        prediction_scores,
        record["ground_truth_polygons"],
        record["ignore_polygons"],
        iou_threshold=iou_threshold,
    )
    true_positives = sum(status == "TP" for status in match_result["statuses"])
    false_positives = sum(status == "FP" for status in match_result["statuses"])
    false_negatives = len(match_result["false_negative_indices"])

    group_counts = {
        name: {"gt": 0, "tp": 0, "fp": 0, "fn": 0}
        for name in ("small", "medium", "large")
    }
    for polygon in record["ground_truth_polygons"]:
        group_counts[_size_group(polygon)]["gt"] += 1
    for match in match_result["matches"]:
        group_name = _size_group(
            record["ground_truth_polygons"][match["ground_truth_index"]]
        )
        group_counts[group_name]["tp"] += 1
    for ground_truth_index in match_result["false_negative_indices"]:
        group_name = _size_group(record["ground_truth_polygons"][ground_truth_index])
        group_counts[group_name]["fn"] += 1

    for prediction_index, status in enumerate(match_result["statuses"]):
        if status != "FP":
            continue
        group_counts[_size_group(predicted_polygons[prediction_index])]["fp"] += 1

    true_positive_confidences = [
        float(prediction_scores[index])
        for index, status in enumerate(match_result["statuses"])
        if status == "TP"
    ]
    false_positive_confidences = [
        float(prediction_scores[index])
        for index, status in enumerate(match_result["statuses"])
        if status == "FP"
    ]

    return {
        "true_positives": int(true_positives),
        "false_positives": int(false_positives),
        "false_negatives": int(false_negatives),
        "ground_truth": len(record["ground_truth_polygons"]),
        "predictions": true_positives + false_positives,
        "ignored_predictions": match_result["ignored_predictions"],
        "decoded_predictions": len(predicted_polygons),
        "decode_stats": decode_stats,
        "group_counts": group_counts,
        "true_positive_confidences": true_positive_confidences,
        "false_positive_confidences": false_positive_confidences,
        "predicted_polygons": predicted_polygons,
        "prediction_scores": prediction_scores,
        "match_result": match_result,
    }


def evaluate_probability_records(
    records: list[dict],
    postprocess: dict | None = None,
    iou_threshold: float = 0.5,
    include_matches: bool = False,
) -> dict:
    postprocess = dict(postprocess or DETECTION_CONFIG["postprocess"])
    totals = {
        "true_positives": 0,
        "false_positives": 0,
        "false_negatives": 0,
        "ground_truth": 0,
        "predictions": 0,
        "ignored_predictions": 0,
        "decoded_predictions": 0,
        "contours_before_filter": 0,
        "score_candidates": 0,
        "polygons_after_filter": 0,
    }
    group_totals = {
        name: {"gt": 0, "tp": 0, "fp": 0, "fn": 0}
        for name in ("small", "medium", "large")
    }
    probability_stats = {name: [] for name in ("mean", "median", "p90", "p95")}
    true_positive_confidences = []
    false_positive_confidences = []
    image_results = []

    for record in records:
        result = _evaluate_one_record(
            record,
            postprocess=postprocess,
            iou_threshold=iou_threshold,
        )
        for key in totals:
            if key in result:
                totals[key] += int(result[key])
        for key in (
            "contours_before_filter",
            "score_candidates",
            "polygons_after_filter",
        ):
            totals[key] += int(result["decode_stats"][key])
        for group_name, counts in result["group_counts"].items():
            for key, value in counts.items():
                group_totals[group_name][key] += int(value)

        true_positive_confidences.extend(result["true_positive_confidences"])
        false_positive_confidences.extend(result["false_positive_confidences"])
        probability_map = np.asarray(record["prob_map"], dtype=np.float32)
        valid_height, valid_width = record.get(
            "valid_size", probability_map.shape
        )
        values = probability_map[:valid_height, :valid_width].reshape(-1)
        if values.size:
            probability_stats["mean"].append(float(values.mean()))
            probability_stats["median"].append(float(np.median(values)))
            probability_stats["p90"].append(float(np.percentile(values, 90)))
            probability_stats["p95"].append(float(np.percentile(values, 95)))

        if include_matches:
            image_results.append(
                {
                    "image_path": record.get("image_path"),
                    "predicted_polygons": result["predicted_polygons"],
                    "prediction_scores": result["prediction_scores"],
                    "match_result": result["match_result"],
                }
            )

    precision, recall, hmean = _metric_triplet(
        totals["true_positives"],
        totals["false_positives"],
        totals["false_negatives"],
    )
    size_metrics = {}
    for group_name, counts in group_totals.items():
        group_precision, group_recall, group_hmean = _metric_triplet(
            counts["tp"], counts["fp"], counts["fn"]
        )
        size_metrics[group_name] = {
            **counts,
            "precision": group_precision,
            "recall": group_recall,
            "f1": group_hmean,
        }

    output = {
        **totals,
        "precision": precision,
        "recall": recall,
        "f1": hmean,
        "size_metrics": size_metrics,
        "probability_stats": {
            key: float(np.mean(values)) if values else 0.0
            for key, values in probability_stats.items()
        },
        "confidence_stats": {
            "true_positive_mean": float(np.mean(true_positive_confidences))
            if true_positive_confidences else 0.0,
            "false_positive_mean": float(np.mean(false_positive_confidences))
            if false_positive_confidences else 0.0,
            "true_positive_count": len(true_positive_confidences),
            "false_positive_count": len(false_positive_confidences),
        },
        "postprocess": postprocess,
    }
    if include_matches:
        output["image_results"] = image_results
    return output


@torch.inference_mode()
def collect_validation_records(
    det_model: nn.Module,
    val_loader: DataLoader,
    scales: tuple[int, ...] = (1024,),
    weights: tuple[float, ...] = (1.0,),
) -> list[dict]:
    if not scales or len(scales) != len(weights) or sum(weights) <= 0:
        raise ValueError("scales và weights phải có cùng độ dài, tổng trọng số phải dương")

    det_model.eval()
    device = next(det_model.parameters()).device
    amp_enabled = device.type == "cuda"
    normalized_weights = [float(weight) / sum(weights) for weight in weights]
    records = []

    for images, ground_truth_batch, ignore_batch, path_batch, valid_sizes in val_loader:
        images = images.to(device, non_blocking=True)
        original_height, original_width = images.shape[-2:]
        fused_probability = torch.zeros(
            (images.shape[0], 1, original_height, original_width),
            dtype=torch.float32,
            device=device,
        )

        for scale, weight in zip(scales, normalized_weights):
            scaled_images = F.interpolate(
                images,
                size=(int(scale), int(scale)),
                mode="bilinear",
                align_corners=False,
            )
            padding = (32 - int(scale) % 32) % 32
            if padding:
                scaled_images = F.pad(
                    scaled_images, (0, padding, 0, padding), value=0.0
                )
            with torch.amp.autocast("cuda", enabled=amp_enabled):
                output = det_model(scaled_images)
            probability = output["prob_map"][:, :, :scale, :scale].float()
            restored = F.interpolate(
                probability,
                size=(original_height, original_width),
                mode="bilinear",
                align_corners=False,
            )
            fused_probability.add_(restored, alpha=weight)

        probability_maps = fused_probability[:, 0].cpu().numpy()
        for index, probability_map in enumerate(probability_maps):
            records.append(
                {
                    "prob_map": probability_map.astype(np.float16),
                    "ground_truth_polygons": ground_truth_batch[index],
                    "ignore_polygons": ignore_batch[index],
                    "image_path": path_batch[index],
                    "valid_size": valid_sizes[index],
                }
            )
    return records


@torch.inference_mode()
def evaluate_dbnet(
    det_model: nn.Module,
    val_loader: DataLoader,
    iou_thresh: float = 0.5,
    scales: tuple[int, ...] = (1024,),
    weights: tuple[float, ...] = (1.0,),
    postprocess: dict | None = None,
) -> dict:
    records = collect_validation_records(
        det_model,
        val_loader,
        scales=scales,
        weights=weights,
    )
    return evaluate_probability_records(
        records,
        postprocess=postprocess,
        iou_threshold=iou_thresh,
    )


def threshold_sweep(
    records: list[dict],
    thresholds: tuple[float, ...] = (0.20, 0.25, 0.30),
    box_thresholds: tuple[float, ...] = (0.50, 0.55, 0.60),
    unclip_ratios: tuple[float, ...] = (1.3, 1.5, 1.7),
) -> dict:
    results = []
    combinations = list(
        itertools.product(thresholds, box_thresholds, unclip_ratios)
    )
    for index, (threshold, box_threshold, unclip_ratio) in enumerate(
        combinations, start=1
    ):
        postprocess = {
            "thresh": float(threshold),
            "box_thresh": float(box_threshold),
            "unclip_ratio": float(unclip_ratio),
        }
        metrics = evaluate_probability_records(
            records,
            postprocess=postprocess,
            iou_threshold=0.5,
        )
        results.append(
            {
                "postprocess": postprocess,
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "true_positives": metrics["true_positives"],
                "false_positives": metrics["false_positives"],
                "false_negatives": metrics["false_negatives"],
            }
        )
        print(
            f"Thử nghiệm {index}/{len(combinations)} | "
            f"thresh={threshold:.2f} | box={box_threshold:.2f} | "
            f"unclip={unclip_ratio:.1f} | H-Mean={metrics['f1'] * 100:.2f}%"
        )

    results.sort(key=lambda item: item["f1"], reverse=True)
    return {
        "best": results[0] if results else None,
        "results": results,
        "evaluated_images": len(records),
        "iou_threshold": 0.5,
    }


def plot_dbnet_history(history: list[dict]):
    if not history:
        return

    epochs = [item["epoch"] for item in history]
    losses = [item["loss"] for item in history]
    precisions = [item["precision"] * 100.0 for item in history]
    recalls = [item["recall"] * 100.0 for item in history]
    f1_scores = [item["f1"] * 100.0 for item in history]

    figure, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].plot(epochs, losses, marker="o", linewidth=2, label="Loss huấn luyện")
    axes[0].set_title("Quá trình hội tụ Loss của DBNet++")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle="--", alpha=0.5)
    axes[0].legend()

    axes[1].plot(epochs, precisions, marker="o", label="Precision")
    axes[1].plot(epochs, recalls, marker="s", label="Recall")
    axes[1].plot(epochs, f1_scores, marker="^", linewidth=2, label="H-Mean")
    axes[1].set_title(f"Validation — H-Mean cao nhất: {max(f1_scores):.2f}%")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Tỷ lệ (%)")
    axes[1].set_ylim(0, 100)
    axes[1].grid(True, linestyle="--", alpha=0.5)
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(WORK_DIR / "detection_training_curves_v3.png", dpi=160, bbox_inches="tight")
    plt.show()


def _capture_rng_state(loader_generator: torch.Generator):
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "loader_generator": loader_generator.get_state(),
    }


def _restore_rng_state(state: dict, loader_generator: torch.Generator):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])
    if state.get("loader_generator") is not None:
        loader_generator.set_state(state["loader_generator"])


def _set_batchnorm_momentum(model: nn.Module, momentum: float = 0.05):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.momentum = float(momentum)


def _make_checkpoint(
    model,
    ema,
    optimizer,
    scaler,
    scheduler,
    epoch,
    global_step,
    best_f1,
    best_epoch,
    epochs_without_improvement,
    history,
    config,
    loader_generator,
    best_ema_state=None,
):
    return {
        "model": model.state_dict(),
        "ema": ema.ema.state_dict(),
        "best_ema": best_ema_state,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": int(epoch),
        "global_step": int(global_step),
        "best_f1": float(best_f1),
        "best_epoch": int(best_epoch),
        "epochs_without_improvement": int(epochs_without_improvement),
        "history": history,
        "config": deepcopy(config),
        "rng_state": _capture_rng_state(loader_generator),
    }


def _load_torch_checkpoint(path, device=None):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def train_dbnet_detector(
    train_items: list[dict],
    val_items: list[dict],
    epochs: int | None = None,
    batch_size: int | None = None,
    gradient_accumulation_steps: int | None = None,
    patience: int | None = None,
    min_delta: float | None = None,
    warmup_epochs: int | None = None,
    base_lr: float | None = None,
    use_dcn: bool | None = None,
    use_asf_spatial: bool | None = None,
    resume_path: str | Path | None = None,
    run_threshold_sweep: bool | None = None,
):
    _clear_cuda_memory()

    config = deepcopy(DETECTION_CONFIG)
    overrides = {
        "epochs": epochs,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "patience": patience,
        "min_delta": min_delta,
        "warmup_epochs": warmup_epochs,
        "base_lr": base_lr,
        "use_dcn": use_dcn,
        "use_asf_spatial": use_asf_spatial,
        "resume_path": str(resume_path) if resume_path is not None else None,
        "run_threshold_sweep": run_threshold_sweep,
    }
    for key, value in overrides.items():
        if value is not None:
            config[key] = value

    if not train_items:
        raise ValueError("Không có dữ liệu Detection để huấn luyện")
    if config["resume_path"] and not Path(config["resume_path"]).is_file():
        raise FileNotFoundError(
            f"Không tìm thấy checkpoint Resume: {config['resume_path']}"
        )
    if not val_items:
        raise ValueError("Không có dữ liệu Validation để chọn checkpoint")
    forbidden_train_splits = {"test", "unseen_test"}
    leaked_train = [
        item for item in train_items
        if str(item.get("split", "train")).lower() in forbidden_train_splits
    ]
    leaked_validation = [
        item for item in val_items
        if str(item.get("split", "val")).lower() in forbidden_train_splits
    ]
    if leaked_train or leaked_validation:
        raise ValueError(
            "Phát hiện dữ liệu Test/Unseen Test trong Train hoặc Validation. "
            "Hãy kiểm tra lại Cell 4 trước khi huấn luyện."
        )
    if config["batch_size"] < 1 or config["gradient_accumulation_steps"] < 1:
        raise ValueError("Batch size và gradient accumulation phải lớn hơn 0")

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Không phát hiện GPU CUDA. Hãy bật GPU trong môi trường Notebook trước khi huấn luyện."
        )

    gpu_count = torch.cuda.device_count()
    device = torch.device("cuda:0")
    amp_enabled = True
    torch.cuda.set_device(device)
    torch.backends.cudnn.benchmark = False

    if gpu_count > 1 and config["batch_size"] < gpu_count:
        config["batch_size"] = gpu_count

    config["device"] = str(device)
    config["gpu_count"] = int(gpu_count)
    config["effective_batch_size"] = (
        config["batch_size"] * config["gradient_accumulation_steps"]
    )
    config["postprocess"] = dict(config["postprocess"])
    config["data_audit"] = deepcopy(globals().get("DET_DATA_AUDIT", {}))

    model = DBNetPlusPlus(
        model_name="resnet50",
        pretrained=config["resume_path"] is None,
        freeze_early=False,
        inner_channels=256,
        k=50,
        dropout=0.15,
        use_dcn=config["use_dcn"],
        use_asf_spatial=config["use_asf_spatial"],
    ).to(device)
    _set_batchnorm_momentum(model, config["batch_norm_momentum"])
    parallel_model = (
        nn.DataParallel(model, device_ids=list(range(gpu_count)))
        if gpu_count > 1
        else model
    )

    optimizer = torch.optim.AdamW(
        model.get_param_groups(
            base_lr=config["base_lr"],
            backbone_lr_mult=config["backbone_lr_multiplier"],
            weight_decay=config["weight_decay"],
        ),
        lr=config["base_lr"],
        weight_decay=config["weight_decay"],
    )

    train_dataset = UnifiedDetectionDataset(
        train_items,
        target_size=config["target_size"],
        is_train=True,
        min_text_size=config["min_text_size"],
    )
    validation_dataset = UnifiedDetectionDataset(
        val_items,
        target_size=config["target_size"],
        is_train=False,
        min_text_size=config["min_text_size"],
    )
    config["target_audit"] = audit_target_generation(
        train_items,
        target_size=config["target_size"],
        sample_limit=48,
    )
    target_audit = config["target_audit"]
    print(
        "Kiểm toán Target Generation trên "
        f"{target_audit['sample_images']} ảnh: "
        f"GT hợp lệ={target_audit['valid_polygons']}, "
        f"shrink chuẩn={target_audit['standard_shrink']}, "
        f"lõi dự phòng={target_audit['fallback_core']}, "
        f"tỷ lệ dự phòng={target_audit['fallback_rate'] * 100:.1f}%"
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)
    num_workers = int(config["num_workers"])
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=amp_enabled,
        drop_last=False,
        persistent_workers=False,
        worker_init_fn=_seed_worker,
        generator=loader_generator,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=max(1, gpu_count),
        shuffle=False,
        num_workers=num_workers,
        pin_memory=amp_enabled,
        collate_fn=det_val_collate,
        persistent_workers=False,
        worker_init_fn=_seed_worker,
    )

    criterion = DBLoss(
        alpha=5.0,
        beta=1.0,
        gamma=10.0,
        negative_ratio=3,
        label_smoothing=0.0,
        min_negative_pixels=100,
    )
    ema = ModelEMA(model, decay=0.995)
    parallel_ema = (
        nn.DataParallel(ema.ema, device_ids=list(range(gpu_count)))
        if gpu_count > 1
        else ema.ema
    )
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    steps_per_epoch = math.ceil(
        len(train_loader) / config["gradient_accumulation_steps"]
    )
    total_steps = max(1, int(config["epochs"]) * steps_per_epoch)
    warmup_steps = min(
        total_steps,
        int(config["warmup_epochs"]) * steps_per_epoch,
    )

    def lr_lambda(step: int):
        if warmup_steps > 0 and step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        decay_steps = max(1, total_steps - warmup_steps)
        progress = min(
            1.0,
            max(0.0, (step - warmup_steps) / decay_steps),
        )
        return (1.0 - progress) ** 0.9

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_lambda,
    )

    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    global BEST_DET_CKPT, BEST_DET_BUNDLE, LAST_DET_CKPT
    if not config["resume_path"]:
        for checkpoint_file in (
            BEST_DET_CKPT,
            BEST_DET_BUNDLE,
            LAST_DET_CKPT,
        ):
            checkpoint_file.unlink(missing_ok=True)
    start_epoch = 1
    global_step = 0
    best_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []
    best_ema_state = None

    if config["resume_path"]:
        resume_checkpoint = _load_torch_checkpoint(
            config["resume_path"], device
        )
        if not all(key in resume_checkpoint for key in ("model", "ema", "optimizer", "scheduler")):
            raise ValueError(
                "resume_path phải trỏ đến dbnetpp_v3_last_checkpoint.pth, "
                "không phải file weights-only."
            )
        stored_config = resume_checkpoint.get("config", {})
        for key in (
            "use_dcn", "use_asf_spatial", "target_size", "batch_size",
            "gradient_accumulation_steps", "base_lr", "warmup_epochs",
            "backbone_lr_multiplier", "epochs", "weight_decay",
            "train_scales", "min_text_size",
        ):
            if key in stored_config and stored_config[key] != config[key]:
                raise ValueError(
                    f"Cấu hình resume không khớp ở {key}: "
                    f"checkpoint={stored_config[key]}, hiện tại={config[key]}"
                )
        model.load_state_dict(resume_checkpoint["model"], strict=True)
        ema.ema.load_state_dict(resume_checkpoint["ema"], strict=True)
        optimizer.load_state_dict(resume_checkpoint["optimizer"])
        scaler.load_state_dict(resume_checkpoint.get("scaler", {}))
        scheduler.load_state_dict(resume_checkpoint["scheduler"])
        start_epoch = int(resume_checkpoint["epoch"]) + 1
        global_step = int(resume_checkpoint.get("global_step", 0))
        best_f1 = float(resume_checkpoint.get("best_f1", -1.0))
        best_epoch = int(resume_checkpoint.get("best_epoch", 0))
        epochs_without_improvement = int(
            resume_checkpoint.get("epochs_without_improvement", 0)
        )
        history = list(resume_checkpoint.get("history", []))
        best_ema_state = resume_checkpoint.get("best_ema")
        if best_ema_state is None:
            best_ema_state = deepcopy(ema.ema.state_dict())
        _restore_rng_state(
            resume_checkpoint.get("rng_state", {}), loader_generator
        )
        print(f"Tiếp tục huấn luyện từ epoch {start_epoch}")
        if start_epoch > int(config["epochs"]):
            raise ValueError(
                "Checkpoint đã hoàn tất số epoch cấu hình. "
                "Không cần chạy lại; nếu muốn huấn luyện thêm, hãy tạo lịch học mới có chủ đích."
            )

    print(
        f"Huấn luyện trên {gpu_count} GPU ({device}) | Train: {len(train_dataset):,} | "
        f"Validation: {len(validation_dataset):,} | "
        f"Batch: {config['batch_size']} × "
        f"Accumulation: {config['gradient_accumulation_steps']} | "
        f"Effective batch: {config['effective_batch_size']} | "
        f"Resolution: {config['target_size']} | Scales: {config['train_scales']}"
    )
    print(
        f"DCNv2: {config['use_dcn']} | Spatial ASF: "
        f"{config['use_asf_spatial']} | LR backbone multiplier: "
        f"{config['backbone_lr_multiplier']}"
    )

    for epoch in range(start_epoch, int(config["epochs"]) + 1):
        parallel_model.train()
        running_loss = 0.0
        loss_components = {"loss_prob": 0.0, "loss_binary": 0.0, "loss_thresh": 0.0}
        batch_count = len(train_loader)
        optimizer.zero_grad(set_to_none=True)

        for batch_index, batch in enumerate(train_loader):
            window_start = (
                batch_index // config["gradient_accumulation_steps"]
            ) * config["gradient_accumulation_steps"]
            window_size = min(
                config["gradient_accumulation_steps"],
                batch_count - window_start,
            )
            is_window_end = (
                (batch_index + 1) % config["gradient_accumulation_steps"] == 0
                or batch_index + 1 == batch_count
            )

            images, shrink_map, training_mask, threshold_map, threshold_mask = batch
            images = images.to(device, non_blocking=True)
            shrink_map = shrink_map.to(device, non_blocking=True)
            training_mask = training_mask.to(device, non_blocking=True)
            threshold_map = threshold_map.to(device, non_blocking=True)
            threshold_mask = threshold_mask.to(device, non_blocking=True)

            (
                images,
                shrink_map,
                training_mask,
                threshold_map,
                threshold_mask,
            ) = _prepare_multiscale_train_batch(
                images,
                shrink_map,
                training_mask,
                threshold_map,
                threshold_mask,
                scales=tuple(config["train_scales"]),
            )

            with torch.amp.autocast("cuda", enabled=amp_enabled):
                predictions = parallel_model(images)

            predictions_fp32 = {
                key: value.float() if torch.is_tensor(value) else value
                for key, value in predictions.items()
            }
            loss = criterion(
                predictions_fp32,
                shrink_map,
                training_mask,
                threshold_map,
                threshold_mask,
            )
            scaled_loss = loss / float(window_size)

            scaler.scale(scaled_loss).backward()
            running_loss += float(loss.detach().item())
            for key, value in criterion.last_metrics.items():
                loss_components[key] += float(value)

            if is_window_end:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                scale_after = scaler.get_scale()
                if scale_after >= scale_before:
                    ema.update(model)
                    scheduler.step()
                    global_step += 1
                optimizer.zero_grad(set_to_none=True)

        average_loss = running_loss / max(1, batch_count)
        average_components = {
            key: value / max(1, batch_count)
            for key, value in loss_components.items()
        }
        validation_metrics = evaluate_dbnet(
            parallel_ema,
            validation_loader,
            iou_thresh=0.5,
            scales=(config["target_size"],),
            weights=(1.0,),
            postprocess=config["postprocess"],
        )
        torch.cuda.empty_cache()

        epoch_record = {
            "epoch": epoch,
            "loss": average_loss,
            **average_components,
            **{
                key: validation_metrics[key]
                for key in (
                    "precision", "recall", "f1", "true_positives",
                    "false_positives", "false_negatives", "ground_truth",
                    "predictions", "ignored_predictions", "decoded_predictions",
                    "contours_before_filter", "score_candidates",
                    "polygons_after_filter", "size_metrics", "probability_stats",
                    "confidence_stats",
                )
            },
            "learning_rate_backbone": float(optimizer.param_groups[0]["lr"]),
            "learning_rate_head": float(optimizer.param_groups[-1]["lr"]),
        }
        history.append(epoch_record)

        print(
            f"Epoch {epoch:03d}/{config['epochs']:03d} | "
            f"Loss: {average_loss:.4f} | "
            f"Precision: {validation_metrics['precision'] * 100:.2f}% | "
            f"Recall: {validation_metrics['recall'] * 100:.2f}% | "
            f"H-Mean: {validation_metrics['f1'] * 100:.2f}% | "
            f"TP/FP/FN: {validation_metrics['true_positives']}/"
            f"{validation_metrics['false_positives']}/"
            f"{validation_metrics['false_negatives']} | "
            f"LR: {optimizer.param_groups[-1]['lr']:.2e}"
        )
        group_metrics = validation_metrics["size_metrics"]
        print(
            "Kích thước chữ — "
            + " | ".join(
                f"{name}: R={group_metrics[name]['recall'] * 100:.1f}% "
                f"F1={group_metrics[name]['f1'] * 100:.1f}% "
                f"GT={group_metrics[name]['gt']}"
                for name in ("small", "medium", "large")
            )
        )

        if validation_metrics["f1"] > best_f1 + config["min_delta"]:
            best_f1 = float(validation_metrics["f1"])
            best_epoch = epoch
            epochs_without_improvement = 0
            best_ema_state = deepcopy(ema.ema.state_dict())
            torch.save(best_ema_state, BEST_DET_CKPT)
            best_bundle = _make_checkpoint(
                model,
                ema,
                optimizer,
                scaler,
                scheduler,
                epoch,
                global_step,
                best_f1,
                best_epoch,
                epochs_without_improvement,
                history,
                config,
                loader_generator,
                best_ema_state=best_ema_state,
            )
            torch.save(best_bundle, BEST_DET_BUNDLE)
        else:
            epochs_without_improvement += 1

        last_bundle = _make_checkpoint(
            model,
            ema,
            optimizer,
            scaler,
            scheduler,
            epoch,
            global_step,
            best_f1,
            best_epoch,
            epochs_without_improvement,
            history,
            config,
            loader_generator,
            best_ema_state=best_ema_state,
        )
        torch.save(last_bundle, LAST_DET_CKPT)

        if epochs_without_improvement >= config["patience"] and epoch > config["warmup_epochs"]:
            print(f"Dừng sớm tại epoch {epoch}; checkpoint tốt nhất ở epoch {best_epoch}")
            break

    if not BEST_DET_CKPT.exists() and best_ema_state is not None:
        torch.save(best_ema_state, BEST_DET_CKPT)
    if not BEST_DET_BUNDLE.exists() and best_ema_state is not None:
        fallback_bundle = _make_checkpoint(
            model,
            ema,
            optimizer,
            scaler,
            scheduler,
            max(0, start_epoch - 1),
            global_step,
            best_f1,
            best_epoch,
            epochs_without_improvement,
            history,
            config,
            loader_generator,
            best_ema_state=best_ema_state,
        )
        fallback_bundle["ema"] = best_ema_state
        torch.save(fallback_bundle, BEST_DET_BUNDLE)
    if not BEST_DET_CKPT.exists() or not BEST_DET_BUNDLE.exists():
        raise RuntimeError("Không tạo được checkpoint tốt nhất")

    if config["run_threshold_sweep"]:
        print("Đang chạy Threshold Sweep trên Validation...")
        best_bundle = _load_torch_checkpoint(BEST_DET_BUNDLE, device)
        sweep_model = DBNetPlusPlus(
            model_name="resnet50",
            pretrained=False,
            freeze_early=False,
            inner_channels=256,
            k=50,
            dropout=0.15,
            use_dcn=config["use_dcn"],
            use_asf_spatial=config["use_asf_spatial"],
        ).to(device)
        sweep_model.load_state_dict(best_bundle["ema"], strict=True)
        parallel_sweep_model = (
            nn.DataParallel(sweep_model, device_ids=list(range(gpu_count)))
            if gpu_count > 1
            else sweep_model
        )
        validation_records = collect_validation_records(
            parallel_sweep_model,
            validation_loader,
            scales=(config["target_size"],),
            weights=(1.0,),
        )
        sweep_result = threshold_sweep(validation_records)
        if sweep_result["best"] is not None:
            BEST_POSTPROCESS_CONFIG = dict(
                sweep_result["best"]["postprocess"]
            )
            best_bundle["config"]["postprocess"] = BEST_POSTPROCESS_CONFIG
            best_bundle["threshold_sweep"] = sweep_result
            torch.save(best_bundle, BEST_DET_BUNDLE)
            with open(
                EXPORT_DIR / "detection_threshold_sweep.json",
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(sweep_result, file, ensure_ascii=False, indent=2)
            print(
                "Cấu hình hậu xử lý tốt nhất trên Validation: "
                f"{BEST_POSTPROCESS_CONFIG}"
            )
        del parallel_sweep_model, sweep_model, validation_records
        _clear_cuda_memory()

    print(f"Checkpoint trọng số tốt nhất: {BEST_DET_CKPT}")
    print(f"Checkpoint Resume: {LAST_DET_CKPT}")
    print(f"H-Mean Validation cao nhất: {best_f1 * 100:.2f}% tại epoch {best_epoch}")
    plot_dbnet_history(history)
    return BEST_DET_CKPT, history


def audit_target_generation(
    items: list[dict],
    target_size: int = 1024,
    sample_limit: int = 48,
) -> dict:
    audit_dataset = UnifiedDetectionDataset(
        items[:sample_limit],
        target_size=target_size,
        is_train=False,
        min_text_size=3.0,
    )
    for item in audit_dataset.items:
        image_path, polygons, ignores = _extract_sample_fields(item)
        image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if image_bgr is None:
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        _, scaled_polygons, valid_height, valid_width = (
            audit_dataset._resize_and_pad(image_rgb, polygons)
        )
        audit_dataset._build_targets(
            scaled_polygons, ignores, valid_height, valid_width
        )

    audit = dict(audit_dataset.target_audit)
    valid_count = max(1, audit["valid_polygons"])
    audit["fallback_rate"] = audit["fallback_core"] / valid_count
    audit["sample_images"] = min(sample_limit, len(items))
    return audit


def _seed_worker(worker_id: int):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


BEST_DET_CKPT, DET_HISTORY = train_dbnet_detector(
    DET_TRAIN_ITEMS,
    DET_VAL_ITEMS,
    use_dcn=DETECTION_CONFIG["use_dcn"],
    use_asf_spatial=DETECTION_CONFIG["use_asf_spatial"],
    resume_path=DETECTION_CONFIG["resume_path"],
)

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F


def _load_detection_bundle(checkpoint_path):
    try:
        return torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(checkpoint_path, map_location="cpu")


def _build_detection_model_from_config(config, device):
    model = DBNetPlusPlus(
        model_name=config.get("model_name", "resnet50"),
        pretrained=False,
        freeze_early=config.get("freeze_early", False),
        inner_channels=int(config.get("inner_channels", 256)),
        k=float(config.get("k", 50)),
        dropout=float(config.get("dropout", 0.15)),
        use_dcn=bool(config.get("use_dcn", False)),
        use_asf_spatial=bool(config.get("use_asf_spatial", False)),
    ).to(device)
    return model


def plot_detection_full_report(
    history: list[dict],
    ckpt_path,
    val_items: list[dict],
    num_samples: int = 6,
    eval_scale: int | None = None,
):
    if not history:
        print("Chưa có lịch sử huấn luyện để trực quan hóa.")
        return

    epochs = np.asarray([item["epoch"] for item in history], dtype=np.int32)
    losses = np.asarray([item["loss"] for item in history], dtype=np.float32)
    precisions = np.asarray(
        [item["precision"] * 100.0 for item in history], dtype=np.float32
    )
    recalls = np.asarray(
        [item["recall"] * 100.0 for item in history], dtype=np.float32
    )
    f1_scores = np.asarray(
        [item["f1"] * 100.0 for item in history], dtype=np.float32
    )

    best_index = int(np.argmax(f1_scores))
    best_epoch = int(epochs[best_index])
    best_f1 = float(f1_scores[best_index])

    figure, axes = plt.subplots(1, 3, figsize=(19, 5.5))
    axes[0].plot(epochs, losses, linewidth=1.8, label="Loss huấn luyện")
    axes[0].set_title("Quá trình hội tụ Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, precisions, label="Precision")
    axes[1].plot(epochs, recalls, label="Recall")
    axes[1].plot(epochs, f1_scores, linewidth=2.2, label="H-Mean")
    axes[1].scatter([best_epoch], [best_f1], s=65, zorder=3)
    axes[1].annotate(
        f"Tốt nhất: {best_f1:.2f}%\nEpoch {best_epoch}",
        (best_epoch, best_f1),
        xytext=(8, -28),
        textcoords="offset points",
        fontsize=9,
    )
    axes[1].set_title("Chỉ số trên Validation")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Tỷ lệ (%)")
    axes[1].set_ylim(0, 100)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(recalls, precisions, marker="o", markersize=3, linewidth=1.0)
    scatter = axes[2].scatter(
        recalls, precisions, c=epochs, cmap="viridis", s=32
    )
    axes[2].scatter(
        [recalls[best_index]],
        [precisions[best_index]],
        marker="*",
        s=160,
        zorder=4,
        label="Checkpoint tốt nhất",
    )
    axes[2].set_title("Cân bằng Precision – Recall")
    axes[2].set_xlabel("Recall (%)")
    axes[2].set_ylabel("Precision (%)")
    axes[2].set_xlim(0, 100)
    axes[2].set_ylim(0, 100)
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    figure.colorbar(scatter, ax=axes[2], label="Epoch")

    figure.tight_layout()
    figure.savefig(
        WORK_DIR / "detection_detailed_metrics_v3.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()

    if ckpt_path is None or not val_items:
        return

    bundle = _load_detection_bundle(ckpt_path)
    if "config" not in bundle or "ema" not in bundle:
        raise ValueError(
            "Cell 8 cần checkpoint bundle có config và EMA; "
            "hãy truyền BEST_DET_BUNDLE thay vì file weights riêng."
        )

    model_config = bundle["config"]
    postprocess = model_config.get(
        "postprocess", DETECTION_CONFIG["postprocess"]
    )
    inference_size = int(
        eval_scale if eval_scale is not None
        else model_config.get("target_size", 960)
    )
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    vis_model = _build_detection_model_from_config(model_config, device)
    vis_model.load_state_dict(bundle["ema"], strict=True)
    vis_model.eval()

    validation_dataset = UnifiedDetectionDataset(
        val_items,
        target_size=int(model_config.get("target_size", 960)),
        is_train=False,
        min_text_size=float(model_config.get("min_text_size", 3.0)),
    )
    sample_count = min(int(num_samples), len(validation_dataset))
    sample_indices = random.Random(SEED).sample(
        range(len(validation_dataset)), k=sample_count
    )
    figure, axes = plt.subplots(
        2,
        sample_count,
        figsize=(5.2 * sample_count, 9),
        squeeze=False,
    )

    amp_enabled = device.type == "cuda"
    with torch.inference_mode():
        for column, sample_index in enumerate(sample_indices):
            (
                image_tensor,
                ground_truth_polygons,
                ignore_polygons,
                image_path,
                valid_height,
                valid_width,
            ) = validation_dataset[sample_index]

            input_tensor = image_tensor.unsqueeze(0).to(device)
            scaled_input = F.interpolate(
                input_tensor,
                size=(inference_size, inference_size),
                mode="bilinear",
                align_corners=False,
            )
            padding = (32 - inference_size % 32) % 32
            if padding:
                scaled_input = F.pad(
                    scaled_input, (0, padding, 0, padding), value=0.0
                )

            with torch.amp.autocast("cuda", enabled=amp_enabled):
                prediction = vis_model(scaled_input)
            probability_map = prediction["prob_map"][:, :, :inference_size, :inference_size]
            probability_map = F.interpolate(
                probability_map.float(),
                size=image_tensor.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )[0, 0].cpu().numpy()
            probability_map = probability_map[:valid_height, :valid_width]

            predicted_polygons, prediction_scores = seg_to_polygons(
                probability_map,
                thresh=float(postprocess["thresh"]),
                box_thresh=float(postprocess["box_thresh"]),
                unclip_ratio=float(postprocess["unclip_ratio"]),
            )
            match_result = _match_image_predictions(
                predicted_polygons,
                prediction_scores,
                ground_truth_polygons,
                ignore_polygons,
                iou_threshold=0.5,
            )

            image = image_tensor[:, :valid_height, :valid_width]
            image = image.permute(1, 2, 0).cpu().numpy()
            image = image * np.asarray([0.229, 0.224, 0.225], dtype=np.float32)
            image = image + np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
            image = np.clip(image, 0.0, 1.0)

            axes[0, column].imshow(image)
            axes[0, column].imshow(
                probability_map, cmap="jet", alpha=0.42, vmin=0.0, vmax=1.0
            )
            axes[0, column].set_title(
                f"Mẫu {sample_index} — Bản đồ xác suất"
            )
            axes[0, column].axis("off")

            axes[1, column].imshow(image)
            for prediction_index, polygon in enumerate(predicted_polygons):
                status = match_result["statuses"][prediction_index]
                if status == "TP":
                    edge_color, line_style = "#00e676", "-"
                elif status == "FP":
                    edge_color, line_style = "#ff1744", "-"
                else:
                    edge_color, line_style = "#9e9e9e", ":"
                axes[1, column].add_patch(
                    patches.Polygon(
                        np.asarray(polygon),
                        closed=True,
                        fill=False,
                        edgecolor=edge_color,
                        linewidth=1.5,
                        linestyle=line_style,
                    )
                )

            for ground_truth_index in match_result["false_negative_indices"]:
                axes[1, column].add_patch(
                    patches.Polygon(
                        np.asarray(ground_truth_polygons[ground_truth_index]),
                        closed=True,
                        fill=False,
                        edgecolor="#ffea00",
                        linewidth=1.6,
                        linestyle="--",
                    )
                )

            for polygon in ignore_polygons:
                axes[1, column].add_patch(
                    patches.Polygon(
                        np.asarray(polygon),
                        closed=True,
                        fill=False,
                        edgecolor="#bdbdbd",
                        linewidth=1.0,
                        linestyle=":",
                    )
                )

            true_positives = sum(
                status == "TP" for status in match_result["statuses"]
            )
            false_positives = sum(
                status == "FP" for status in match_result["statuses"]
            )
            false_negatives = len(match_result["false_negative_indices"])
            axes[1, column].set_title(
                f"TP: {true_positives} | FP: {false_positives} | FN: {false_negatives}"
            )
            axes[1, column].axis("off")

    figure.tight_layout()
    figure.savefig(
        WORK_DIR / "detection_val_predictions_v3.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()

    print(f"Checkpoint trực quan hóa: {ckpt_path}")
    print(f"Kích thước suy luận: {inference_size} × {inference_size}")
    print(f"Tham số hậu xử lý: {postprocess}")


plot_detection_full_report(
    history=DET_HISTORY,
    ckpt_path=BEST_DET_BUNDLE,
    val_items=DET_VAL_ITEMS,
    num_samples=6,
    eval_scale=DETECTION_CONFIG["target_size"],
)
